# Enriquecimiento de personas (dirección y guion) — corpus completo

Objetivo: obtener `sexo` y `país` para cada director/a y guionista del
corpus (~2052 películas), partiendo de `icaa_peliculas.csv`.

**Fuentes de crédito** (quién dirigió/escribió cada película):
- ICAA (`directores_icaa`/`guionistas_icaa`) — para las ~1775 películas con `icaa_id`
- TMDB directo — para las ~277 películas SIN `icaa_id` (sección aislada, no se mezcla con lo anterior)

**Fuentes de enriquecimiento de persona** (sexo/país), en cascada:
1. TMDB (`search/person` + ficha de persona)
2. Wikidata (vía el `imdb_id` que da TMDB en `external_ids`)
3. IMDb directo — solo país (campo estructurado, con respaldo de biografía en prosa)
4. Correcciones manuales (reutilizadas de la fase anterior del proyecto)

## 1. Librerías y configuración

In [2]:
import os
import re
import json
import csv
import time
import random
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

BASE = Path("..")
CSV_DIR = BASE / "3 - csv"

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
assert TMDB_API_KEY, "No se encontró TMDB_API_KEY en el .env"

print("✓ Librerías y configuración cargadas")


✓ Librerías y configuración cargadas


In [3]:
def get_session():
    s = requests.Session()
    s.headers.update({"User-Agent": "ProyectoAcademicoLatam/1.0"})
    return s

session = get_session()


def cargar_cache_json(path):
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}


def guardar_cache_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)


def normalizar_texto(texto):
    import unicodedata
    if not isinstance(texto, str):
        return ""
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t).strip()

print("✓ Utilidades de caché y normalización listas")


✓ Utilidades de caché y normalización listas


In [4]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager


def iniciar_driver():
    options = FirefoxOptions()
    options.add_argument("--headless")
    driver = webdriver.Firefox(
        service=FirefoxService(GeckoDriverManager().install()),
        options=options,
    )
    driver.set_page_load_timeout(20)
    return driver

print("✓ iniciar_driver definido (Firefox headless, igual que en notebook 1)")


✓ iniciar_driver definido (Firefox headless, igual que en notebook 1)


## 2. Cargar corpus

In [5]:
icaa_peliculas = pd.read_csv(CSV_DIR / "icaa_peliculas.csv", sep=';')
print(f"Total películas: {len(icaa_peliculas)}")
print(f"Con icaa_id: {icaa_peliculas['icaa_id'].notna().sum()}")
print(f"Sin icaa_id: {icaa_peliculas['icaa_id'].isna().sum()}")
icaa_peliculas.head()


Total películas: 2052
Con icaa_id: 1775
Sin icaa_id: 277


,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio,anio_reposicion,titulo_busqueda,articulo,icaa_id,...,guionistas_icaa,subvenciones_icaa,subvenciones_total,empresas_productoras_icaa,recaudacion_total,espectadores_total,anios_en_cartelera,num_anios,distribuidora_final,distribuidora_fuente
0,"091, policia al habla (1960) (re)",Independent,16/05/2025,42.0,12,2025,1960.0,"091, policia al habla",NaN,NaN,...,NaN,NaN,NaN,NaN,42.0,12,2025,1,Independent,pdf_heuristica
1,Todos lo saben,Universal,14/09/2018,83.0,18,2022,NaN,Todos lo saben,NaN,100317.0,...,"[""Asghar Farhadi""]","[{""concepto"": ""Ayudas Generales para la produc...",280000.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 50.0, ""...",83.0,18,2022,1,"UNIVERSAL PICTURES INTERNATIONAL SPAIN, S.L. (...",ficha_icaa
2,"Generacion silenciosa, La",Independent,10/09/2021,129.0,35,2021,NaN,Generacion silenciosa,La,100421.0,...,"[""Ferrán Navarro-Beltrán""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",129.0,35,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
3,Platon,Independent,08/10/2021,237.0,60,2021,NaN,Platon,NaN,100517.0,...,"[""Iván López González""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",237.0,60,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
4,Agua,Independent,11/02/2022,1975.0,319,2022,NaN,Agua,NaN,100820.0,...,"[""Vicente Pérez Herrero"", ""Marta Martínez Rodr...",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",2296.0,357,"2022,2023",2,LAS PRODUCCIONES DE LOS IMAGINARIOS. PRODUCCIO...,ficha_icaa


## 3. Sección aislada — créditos vía TMDB directo (solo películas sin `icaa_id`)

Estas películas no tienen `directores_icaa`/`guionistas_icaa` porque ICAA
nunca las resolvió. En vez de dejarlas fuera del análisis de personas, se
buscan directamente en TMDB por título+año, y se extraen sus créditos de
`director`/`guionista` con `person_id` ya resuelto (más fiable que buscar
por nombre, porque el endpoint de créditos da el id directo).

Se guarda en una tabla separada, marcada `fuente_credito='tmdb_directo'`,
para que no se confunda con lo verificado por ICAA.

In [6]:
sin_icaa_id = icaa_peliculas[icaa_peliculas['icaa_id'].isna()].copy()
print(f"Películas sin icaa_id a buscar en TMDB: {len(sin_icaa_id)}")
sin_icaa_id[['titulo', 'anio', 'fecha_estreno']].head()


Películas sin icaa_id a buscar en TMDB: 277


,titulo,anio,fecha_estreno
0,"091, policia al habla (1960) (re)",2025,16/05/2025
152,117,2025,12/06/2025
826,"2 Años, 4 Meses (2 Urte, 4 Hilabete)",2021,04/09/2020
1112,33 años de oscuridad,2022,23/05/2022
1779,A dos velas,2025,18/02/2025


In [7]:
def buscar_pelicula_tmdb(titulo, anio, session, timeout=15):
    """
    Busca una película en TMDB por título + año, con margen de +/-1 año
    (la calificación puede caer en un año distinto al de producción,
    igual que vimos con ICAA). Devuelve (tmdb_id, titulo_tmdb) o (None, None).
    """
    url = "https://api.themoviedb.org/3/search/movie"
    intentos_anio = [anio, anio - 1, anio + 1] if pd.notna(anio) else [None]
    for anio_intento in intentos_anio:
        params = {"api_key": TMDB_API_KEY, "language": "es-ES", "query": titulo}
        if anio_intento:
            params["year"] = int(anio_intento)
        try:
            r = session.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            resultados = r.json().get("results", [])
        except Exception as e:
            print(f"  Error búsqueda TMDB '{titulo}': {e}")
            continue
        if resultados:
            top = resultados[0]
            return top.get("id"), top.get("title")
    return None, None

print("✓ buscar_pelicula_tmdb definida")


✓ buscar_pelicula_tmdb definida


In [8]:
PELICULA_TMDB_SEARCH_CACHE = CSV_DIR / "pelicula_tmdb_search_cache.json"

pelicula_search_cache = cargar_cache_json(PELICULA_TMDB_SEARCH_CACHE)
print(f"Caché existente: {len(pelicula_search_cache)} películas ya buscadas")

pendientes_pelicula = [
    row for row in sin_icaa_id.itertuples()
    if row.titulo not in pelicula_search_cache
]
print(f"Pendientes: {len(pendientes_pelicula)}")

for i, row in enumerate(pendientes_pelicula, 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(pendientes_pelicula)}] {row.titulo}")
    tmdb_id, titulo_tmdb = buscar_pelicula_tmdb(row.titulo, row.anio, session)
    pelicula_search_cache[row.titulo] = {"tmdb_id": tmdb_id, "titulo_tmdb": titulo_tmdb}
    if i % 25 == 0:
        guardar_cache_json(PELICULA_TMDB_SEARCH_CACHE, pelicula_search_cache)
    time.sleep(0.25)

guardar_cache_json(PELICULA_TMDB_SEARCH_CACHE, pelicula_search_cache)

sin_icaa_id['tmdb_id'] = sin_icaa_id['titulo'].map(
    lambda t: pelicula_search_cache.get(t, {}).get("tmdb_id")
)
print(f"\n✓ Con tmdb_id encontrado: {sin_icaa_id['tmdb_id'].notna().sum()} / {len(sin_icaa_id)}")


Caché existente: 0 películas ya buscadas
Pendientes: 277
  [25/277] Aun
  [50/277] Chichi y yo
  [75/277] Dies d'estiu i de pluja
  [100/277] Flores de tierra quemada. Estrategias del terror en Guatemala y España
  [125/277] Invasion pequeña
  [150/277] Metralla de amor
  [175/277] Nosotros no nos mataremos con pistolas
  [200/277] Quieres salir puedes entrar
  [225/277] Sexo, drogas, rock 'n' roll y política. Instituto Santamarca, 1975-1985
  [250/277] Tu me abrasas
  [275/277] Zoo, Sobreviure a l'incendi

✓ Con tmdb_id encontrado: 120 / 277


### Persistir `tmdb_id` en `icaa_peliculas.csv`

Así cada película queda resoluble por un identificador estable
(`icaa_id` o `tmdb_id`, nunca los dos a la vez) sin depender de volver a
buscar por título en ningún notebook posterior.

In [9]:
icaa_peliculas['tmdb_id'] = icaa_peliculas['titulo'].map(
    lambda t: pelicula_search_cache.get(t, {}).get("tmdb_id")
)
# Solo debería quedar poblado para las que no tienen icaa_id -- si alguna
# fila tiene ambos, algo salió mal en el filtro de más arriba.
ambos = icaa_peliculas[icaa_peliculas['icaa_id'].notna() & icaa_peliculas['tmdb_id'].notna()]
assert len(ambos) == 0, f"{len(ambos)} filas con icaa_id Y tmdb_id a la vez -- revisar"

icaa_peliculas.to_csv(CSV_DIR / "icaa_peliculas.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(f"✓ icaa_peliculas.csv actualizado con tmdb_id")
print(f"  Con icaa_id: {icaa_peliculas['icaa_id'].notna().sum()}")
print(f"  Con tmdb_id (sin icaa_id): {icaa_peliculas['tmdb_id'].notna().sum()}")
print(f"  Sin ninguno de los dos: {(icaa_peliculas['icaa_id'].isna() & icaa_peliculas['tmdb_id'].isna()).sum()}")


✓ icaa_peliculas.csv actualizado con tmdb_id
  Con icaa_id: 1775
  Con tmdb_id (sin icaa_id): 120
  Sin ninguno de los dos: 157


In [10]:
# También actualizar icaa_peliculas en MySQL -- si no, queda desincronizada
# con el CSV (sin tmdb_id, sin las 277 filas que ahora sí tienen identificador)
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)
icaa_peliculas.to_sql('icaa_peliculas', engine, if_exists='replace', index=False)
engine.dispose()
print("✓ icaa_peliculas actualizado en MySQL (con tmdb_id incluido)")

✓ icaa_peliculas actualizado en MySQL (con tmdb_id incluido)


In [11]:
GENERO_MAP = {0: None, 1: "mujer", 2: "hombre", 3: "no_binario"}
JOBS_GUION = {"Screenplay", "Writer", "Story", "Teleplay", "Co-Writer"}


def obtener_credits_tmdb(tmdb_id, session, timeout=15):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/credits"
    params = {"api_key": TMDB_API_KEY, "language": "es-ES"}
    try:
        r = session.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r.json().get("crew", [])
    except Exception as e:
        print(f"  Error credits tmdb_id={tmdb_id}: {e}")
        return []

print("✓ obtener_credits_tmdb definida (reutilizada de la fase anterior)")


✓ obtener_credits_tmdb definida (reutilizada de la fase anterior)


In [12]:
CREDITS_TMDB_DIRECTO_CACHE = CSV_DIR / "credits_tmdb_directo_cache.csv"

if CREDITS_TMDB_DIRECTO_CACHE.exists():
    credits_tmdb_directo = pd.read_csv(CREDITS_TMDB_DIRECTO_CACHE, sep=';')
    tmdb_ids_hechos = set(credits_tmdb_directo['tmdb_id'].dropna().unique())
    print(f"Reanudando: {len(tmdb_ids_hechos)} películas ya procesadas")
else:
    credits_tmdb_directo = pd.DataFrame()
    tmdb_ids_hechos = set()
    print("Empezando desde cero")

tmdb_ids_a_consultar = sorted({
    tid for tid in sin_icaa_id['tmdb_id'].dropna().unique()
    if tid not in tmdb_ids_hechos
})
print(f"Pendientes: {len(tmdb_ids_a_consultar)}")

registros = []
for i, tmdb_id in enumerate(tmdb_ids_a_consultar, 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(tmdb_ids_a_consultar)}] tmdb_id={tmdb_id}")

    crew = obtener_credits_tmdb(tmdb_id, session)
    for miembro in crew:
        department = miembro.get("department")
        job = miembro.get("job")
        if department == "Directing" and job == "Director":
            rol = "director"
        elif department == "Writing" and job in JOBS_GUION:
            rol = "guionista"
        else:
            continue
        registros.append({
            "icaa_id": None,
            "tmdb_id": tmdb_id,
            "rol": rol,
            "nombre": miembro.get("name"),
            "person_id_tmdb": miembro.get("id"),
            "fuente_credito": "tmdb_directo",
        })

    if i % 50 == 0 and registros:
        batch = pd.DataFrame(registros)
        credits_tmdb_directo = pd.concat([credits_tmdb_directo, batch], ignore_index=True)
        credits_tmdb_directo.to_csv(CREDITS_TMDB_DIRECTO_CACHE, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
        registros = []
        print("  ✓ Guardado parcial")

    time.sleep(0.25)

if registros:
    batch = pd.DataFrame(registros)
    credits_tmdb_directo = pd.concat([credits_tmdb_directo, batch], ignore_index=True)
    credits_tmdb_directo.to_csv(CREDITS_TMDB_DIRECTO_CACHE, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)

print(f"\n✓ Créditos vía TMDB directo: {len(credits_tmdb_directo)} filas")
if len(credits_tmdb_directo):
    print(credits_tmdb_directo['rol'].value_counts())


Empezando desde cero
Pendientes: 120
  [25/120] tmdb_id=811631.0
  [50/120] tmdb_id=1033028.0
  ✓ Guardado parcial
  [75/120] tmdb_id=1229067.0
  [100/120] tmdb_id=1386565.0
  ✓ Guardado parcial

✓ Créditos vía TMDB directo: 275 filas
rol
director     145
guionista    130
Name: count, dtype: int64


## 4. Flujo principal — créditos desde ICAA (películas con `icaa_id`)

`directores_icaa`/`guionistas_icaa` vienen como listas JSON serializadas
(una por película). Se explotan a formato largo (una fila por crédito) y
se limpian los nombres antes de seguir.

In [13]:
def limpiar_nombre_credito(nombre_crudo):
    """
    Limpia un nombre tal como viene de la ficha ICAA:
    1. Quita anotaciones de segmento (": 'Título del segmento'.") --
       películas de episodios/antología con un director distinto por segmento.
    2. Separa casos donde dos personas quedaron concatenadas sin separador
       en el parser de notebook 1 ("Apellido1, Nombre1 Apellido2, Nombre2").
    3. Normaliza "Apellido, Nombre" -> "Nombre Apellido" (ICAA mezcla ambos
       formatos dentro del mismo campo, sin consistencia).
    Devuelve una LISTA de nombres limpios (normalmente 1, a veces 2 si había
    concatenación).
    """
    texto = nombre_crudo.strip()

    texto = re.sub(r":\s*['\u2018\u2019].*?['\u2018\u2019]\.?\s*$", "", texto).strip()
    texto = texto.rstrip(".").strip()

    partes_coma = texto.split(",")
    if len(partes_coma) == 3:
        apellido1 = partes_coma[0].strip()
        resto = partes_coma[1].strip().split()
        nombre2 = partes_coma[2].strip()
        if len(resto) >= 2:
            nombre1 = resto[0]
            apellido2 = " ".join(resto[1:])
            return [f"{nombre1} {apellido1}", f"{nombre2} {apellido2}"]

    if len(partes_coma) == 2:
        apellido, nombre = partes_coma[0].strip(), partes_coma[1].strip()
        return [f"{nombre} {apellido}"]

    return [texto]

print("✓ limpiar_nombre_credito definida (probada contra los ~15 casos reales detectados)")


✓ limpiar_nombre_credito definida (probada contra los ~15 casos reales detectados)


In [14]:
con_icaa_id = icaa_peliculas[icaa_peliculas['icaa_id'].notna()].copy()

registros_icaa = []
for _, row in con_icaa_id.iterrows():
    for col, rol in (('directores_icaa', 'director'), ('guionistas_icaa', 'guionista')):
        valor = row[col]
        if pd.isna(valor):
            continue
        try:
            nombres_crudos = json.loads(valor)
        except Exception:
            continue
        for nombre_crudo in nombres_crudos:
            for nombre_limpio in limpiar_nombre_credito(nombre_crudo):
                registros_icaa.append({
                    "icaa_id": row['icaa_id'],
                    "tmdb_id": None,
                    "rol": rol,
                    "nombre": nombre_limpio,
                    "person_id_tmdb": None,
                    "fuente_credito": "icaa",
                })

credits_icaa = pd.DataFrame(registros_icaa)
print(f"✓ Créditos ICAA explotados y limpiados: {len(credits_icaa)} filas")
print(credits_icaa['rol'].value_counts())
credits_icaa.head()


✓ Créditos ICAA explotados y limpiados: 4884 filas
rol
guionista    2867
director     2017
Name: count, dtype: int64


,icaa_id,tmdb_id,rol,nombre,person_id_tmdb,fuente_credito
0,100317.0,None,director,Asghar Farhadi,None,icaa
1,100317.0,None,guionista,Asghar Farhadi,None,icaa
2,100421.0,None,director,Ferrán Navarro-Beltrán,None,icaa
3,100421.0,None,guionista,Ferrán Navarro-Beltrán,None,icaa
4,100517.0,None,director,Iván López González,None,icaa


## 5. Unificar créditos (ICAA + TMDB directo)

In [15]:
credits_df = pd.concat([credits_icaa, credits_tmdb_directo], ignore_index=True)
print(f"✓ Total créditos unificados: {len(credits_df)}")
print("\nPor rol:")
print(credits_df['rol'].value_counts())
print("\nPor fuente:")
print(credits_df['fuente_credito'].value_counts())


✓ Total créditos unificados: 5159

Por rol:
rol
guionista    2997
director     2162
Name: count, dtype: int64

Por fuente:
fuente_credito
icaa            4884
tmdb_directo     275
Name: count, dtype: int64


## 6. Resolver `person_id` en TMDB

Para los créditos de ICAA (que solo traen el nombre), buscar la persona por
nombre en TMDB. Los de `tmdb_directo` ya traen `person_id_tmdb` resuelto
directamente del endpoint de créditos -- más fiable, no hace falta buscar.

In [16]:
def buscar_persona_tmdb_por_nombre(nombre, session, timeout=15):
    """Busca una persona en TMDB por nombre. Devuelve (person_id, nombre_tmdb) o (None, None)."""
    url = "https://api.themoviedb.org/3/search/person"
    params = {"api_key": TMDB_API_KEY, "language": "es-ES", "query": nombre}
    try:
        r = session.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        resultados = r.json().get("results", [])
        if not resultados:
            return None, None
        top = resultados[0]
        return top.get("id"), top.get("name")
    except Exception as e:
        print(f"  Error búsqueda persona '{nombre}': {e}")
        return None, None

print("✓ buscar_persona_tmdb_por_nombre definida (reutilizada)")


✓ buscar_persona_tmdb_por_nombre definida (reutilizada)


In [17]:
PERSON_SEARCH_CACHE = CSV_DIR / "person_search_tmdb_cache.json"

person_search_cache = cargar_cache_json(PERSON_SEARCH_CACHE)

necesitan_match = credits_df[credits_df['person_id_tmdb'].isna()]
nombres_unicos = necesitan_match['nombre'].dropna().unique()
print(f"Nombres únicos a buscar en TMDB: {len(nombres_unicos)}")

pendientes_nombres = [n for n in nombres_unicos if n not in person_search_cache]
print(f"Pendientes: {len(pendientes_nombres)}")

for i, nombre in enumerate(pendientes_nombres, 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(pendientes_nombres)}]")
    person_id, nombre_tmdb = buscar_persona_tmdb_por_nombre(nombre, session)
    person_search_cache[nombre] = {"person_id": person_id, "nombre_tmdb": nombre_tmdb}
    if i % 25 == 0:
        guardar_cache_json(PERSON_SEARCH_CACHE, person_search_cache)
    time.sleep(0.25)

guardar_cache_json(PERSON_SEARCH_CACHE, person_search_cache)
print("✓ Búsqueda de persona por nombre completada")


Nombres únicos a buscar en TMDB: 2591
Pendientes: 2591
  [25/2591]
  [50/2591]
  [75/2591]
  [100/2591]
  [125/2591]
  [150/2591]
  [175/2591]
  [200/2591]
  [225/2591]
  [250/2591]
  [275/2591]
  [300/2591]
  [325/2591]
  [350/2591]
  [375/2591]
  [400/2591]
  [425/2591]
  [450/2591]
  [475/2591]
  [500/2591]
  [525/2591]
  [550/2591]
  [575/2591]
  [600/2591]
  [625/2591]
  [650/2591]
  [675/2591]
  [700/2591]
  [725/2591]
  [750/2591]
  [775/2591]
  [800/2591]
  [825/2591]
  [850/2591]
  [875/2591]
  [900/2591]
  [925/2591]
  [950/2591]
  [975/2591]
  [1000/2591]
  [1025/2591]
  [1050/2591]
  [1075/2591]
  [1100/2591]
  [1125/2591]
  [1150/2591]
  [1175/2591]
  [1200/2591]
  [1225/2591]
  [1250/2591]
  [1275/2591]
  [1300/2591]
  [1325/2591]
  [1350/2591]
  [1375/2591]
  [1400/2591]
  [1425/2591]
  [1450/2591]
  [1475/2591]
  [1500/2591]
  [1525/2591]
  [1550/2591]
  [1575/2591]
  [1600/2591]
  [1625/2591]
  [1650/2591]
  [1675/2591]
  [1700/2591]
  [1725/2591]
  [1750/2591]
  [1775

In [18]:
def resolver_person_id(row):
    if pd.notna(row.get('person_id_tmdb')):
        return row['person_id_tmdb'], 'directo'
    match = person_search_cache.get(row['nombre'], {})
    pid = match.get('person_id')
    if pid is None:
        return None, None
    coincide = normalizar_texto(match.get('nombre_tmdb')) == normalizar_texto(row['nombre'])
    return pid, ('busqueda_nombre_exacta' if coincide else 'busqueda_nombre_aproximada')

resueltos = credits_df.apply(resolver_person_id, axis=1, result_type='expand')
credits_df['person_id'] = resueltos[0]
credits_df['metodo_match_persona'] = resueltos[1]

print(credits_df['metodo_match_persona'].value_counts(dropna=False))
print(f"\nCréditos con person_id resuelto: {credits_df['person_id'].notna().sum()} / {len(credits_df)}")
print("⚠ Revisar manualmente los 'busqueda_nombre_aproximada' -- el match no es exacto")


metodo_match_persona
busqueda_nombre_exacta        3935
NaN                            637
busqueda_nombre_aproximada     312
directo                        275
Name: count, dtype: int64

Créditos con person_id resuelto: 4522 / 5159
⚠ Revisar manualmente los 'busqueda_nombre_aproximada' -- el match no es exacto


In [19]:
credits_df[credits_df['metodo_match_persona'] == 'busqueda_nombre_aproximada']


,icaa_id,tmdb_id,rol,nombre,person_id_tmdb,fuente_credito,person_id,metodo_match_persona
13,100920.0,None,guionista,José Ramón Fernández,None,icaa,4748222.0,busqueda_nombre_aproximada
14,101120.0,None,director,César Souto,None,icaa,2434913.0,busqueda_nombre_aproximada
15,101120.0,None,guionista,César Souto,None,icaa,2434913.0,busqueda_nombre_aproximada
16,101120.0,None,guionista,Cristina Liz,None,icaa,2434919.0,busqueda_nombre_aproximada
17,101222.0,None,director,Eloy Calvo Campos,None,icaa,2839650.0,busqueda_nombre_aproximada
...,...,...,...,...,...,...,...,...
4783,96020.0,None,guionista,Alba Cros Pellisé,None,icaa,1524680.0,busqueda_nombre_aproximada
4837,98520.0,None,director,Alejandro Alvarado Jódar,None,icaa,1172340.0,busqueda_nombre_aproximada
4838,98520.0,None,director,Concha Barquero Artés,None,icaa,1172339.0,busqueda_nombre_aproximada
4839,98520.0,None,guionista,Alejandro Alvarado Jódar,None,icaa,1172340.0,busqueda_nombre_aproximada


### Revisiones manuales

In [20]:
aproximados_unicos = (
    credits_df[credits_df['metodo_match_persona'] == 'busqueda_nombre_aproximada']
    [['nombre', 'person_id']]
    .drop_duplicates()
    .sort_values('nombre')
)
print(f"# {len(aproximados_unicos)} nombres únicos con match aproximado -- rellena el person_id "
      "correcto o deja None para descartar el match (la persona queda sin sexo/país)")
print("OVERRIDES_PERSON_SEARCH = {")
for _, fila in aproximados_unicos.iterrows():
    nombre_icaa = fila['nombre']
    match = person_search_cache.get(nombre_icaa, {})
    nombre_tmdb = match.get('nombre_tmdb')
    print(f"    {nombre_icaa!r}: None,  # TMDB propuso: {nombre_tmdb!r} (person_id={int(fila['person_id'])})")
print("}")

# 192 nombres únicos con match aproximado -- rellena el person_id correcto o deja None para descartar el match (la persona queda sin sexo/país)
OVERRIDES_PERSON_SEARCH = {
    'Adolfo Dufour Andia': None,  # TMDB propuso: 'Adolfo Dufour' (person_id=1413007)
    'Agustí Villaronga Riutort': None,  # TMDB propuso: 'Agustí Villaronga' (person_id=151162)
    'Aina Calleja Cortés': None,  # TMDB propuso: 'Aina Calleja' (person_id=1182523)
    'Aitor González De Langarica': None,  # TMDB propuso: 'Aitor González de Langarica Mendizabal' (person_id=3235428)
    'Aizea Roca': None,  # TMDB propuso: 'Aizea Roca Berridi' (person_id=4999905)
    'Alba Cros Pellisé': None,  # TMDB propuso: 'Alba Cros' (person_id=1524680)
    'Albert Bertran': None,  # TMDB propuso: 'Albert Bertran Bas' (person_id=4044587)
    'Albert Solé Bruset': None,  # TMDB propuso: 'Albert Solé' (person_id=1088457)
    'Alberto Armas': None,  # TMDB propuso: 'Alberto Armas Díaz' (person_id=2998049)
    'Alberto Rodríguez': No

In [21]:
# 192 nombres únicos con match aproximado -- rellena el person_id correcto o deja None para descartar el match (la persona queda sin sexo/país)
OVERRIDES_PERSON_SEARCH = {
    'Adolfo Dufour Andia': None,  # TMDB propuso: 'Adolfo Dufour' (person_id=1413007)
    'Agustí Villaronga Riutort': None,  # TMDB propuso: 'Agustí Villaronga' (person_id=151162)
    'Aina Calleja Cortés': None,  # TMDB propuso: 'Aina Calleja' (person_id=1182523)
    'Aitor González De Langarica': None,  # TMDB propuso: 'Aitor González de Langarica Mendizabal' (person_id=3235428)
    'Aizea Roca': None,  # TMDB propuso: 'Aizea Roca Berridi' (person_id=4999905)
    'Alba Cros Pellisé': None,  # TMDB propuso: 'Alba Cros' (person_id=1524680)
    'Albert Bertran': None,  # TMDB propuso: 'Albert Bertran Bas' (person_id=4044587)
    'Albert Solé Bruset': None,  # TMDB propuso: 'Albert Solé' (person_id=1088457)
    'Alberto Armas': None,  # TMDB propuso: 'Alberto Armas Díaz' (person_id=2998049)
    'Alberto Rodríguez': 6521,  # TMDB propuso: 'Alberto Mimbrera Rodríguez' (person_id=5652588)
    'Alberto Vázquez Rico': None,  # TMDB propuso: 'Alberto Vázquez' (person_id=1515828)
    'Alejandro Alvarado Jódar': None,  # TMDB propuso: 'Alejandro Alvarado' (person_id=1172340)
    'Alejandro Blasco': None,  # TMDB propuso: 'Alejandro Marzoa Blasco' (person_id=1161119)
    'Alejandro Mateos': None,  # TMDB propuso: 'Alex Mateos' (person_id=4346432)
    'Alexandre Cirici-Pellicer': None,  # TMDB propuso: 'Alexandre Cirici Pellicer' (person_id=1877869)
    'Alfonso Suarez Suarez': None,  # TMDB propuso: 'Alfonso Suárez' (person_id=1121370)
    'Alfonso Suárez Suárez': None,  # TMDB propuso: 'Alfonso Suárez' (person_id=1121370)
    'Alfred Pérez Fargas': None,  # TMDB propuso: 'Alfred Pérez-Fargas' (person_id=1489447)
    'Alfredo Carrasco': None,  # TMDB propuso: 'Alfredo Carrasco Alonso' (person_id=3574547)
    'Ana Lambarri Tellaeche': None,  # TMDB propuso: 'Ana Lambarri' (person_id=2836489)
    'Andres Koppel': None,  # TMDB propuso: 'Andrés M. Koppel' (person_id=67263)
    'Angeles González Sinde': None,  # TMDB propuso: 'Ángeles González-Sinde' (person_id=230209)
    'Antonio Gimenez-Rico': None,  # TMDB propuso: 'Antonio Giménez Rico' (person_id=565852)
    'Antonio Giménez-Rico': None,  # TMDB propuso: 'Antonio Giménez Rico' (person_id=565852)
    'Antonio Mas-Guindal': None,  # TMDB propuso: 'Antonio Mas Guindal' (person_id=1173027)
    'Antonio Mercero Santos': None,  # TMDB propuso: 'Santos Mercero' (person_id=1451411)
    'Arantxa Aguirre Carballeira': None,  # TMDB propuso: 'Arantxa Aguirre' (person_id=1047274)
    'Ariadna Fortuny Cardona': None,  # TMDB propuso: 'Ariadna Fortuny' (person_id=3960226)
    'Arturo Dueñas Herrero': None,  # TMDB propuso: 'Arturo Dueñas' (person_id=2826886)
    'Arturo Pérez Reverte': None,  # TMDB propuso: 'Arturo Pérez-Reverte' (person_id=8908)
    'Augusto Martínez Torres': None,  # TMDB propuso: 'Augusto M. Torres' (person_id=37515)
    'Azucena Rodríguez': None,  # TMDB propuso: 'Azucena Carbajosa Rodriguez' (person_id=5421335)
    'Belén Sánchez-Arévalo': None,  # TMDB propuso: 'Belén Sánchez Arévalo' (person_id=2406756)
    'Beñat Gereka': None,  # TMDB propuso: 'Beñat Gereka Bikuña' (person_id=5718669)
    'Bianca Franceza': None,  # TMDB propuso: 'Bianca Franceza Omonte' (person_id=4543717)
    'Burnin´ Percebes': None,  # TMDB propuso: "Burnin' Percebes" (person_id=1395595)
    'Carlos Marqués Marcet': None,  # TMDB propuso: 'Carlos Marques-Marcet' (person_id=1288697)
    'Carlos Therón': None,  # TMDB propuso: 'Carlos Therón Sánchez' (person_id=588816)
    'Carmen López-Areal': None,  # TMDB propuso: 'Carmen López Areal' (person_id=2033131)
    'Carmina Balaguer Montón': None,  # TMDB propuso: 'Carmina Balaguer' (person_id=3698547)
    'Casandra Macías': None,  # TMDB propuso: 'Casandra Macías Gago' (person_id=1360059)
    'Celia Rico': None,  # TMDB propuso: 'Celia Rico Clavellino' (person_id=1384823)
    'Chumilla-Carbajosa': None,  # TMDB propuso: 'Juan Manuel Chumilla-Carbajosa' (person_id=1168983)
    'Clara Martínez-Lázaro': None,  # TMDB propuso: 'Clara Martínez-Lázaro Alameda' (person_id=1521812)
    'Concha Barquero Artés': None,  # TMDB propuso: 'Concha Barquero' (person_id=1172339)
    'Cris Liz': None,  # TMDB propuso: 'Cris Lizarraga' (person_id=4266018)
    'Cristina Hernández-Carrillo': None,  # TMDB propuso: 'Cristina Hernández-Carrillo de la Higuera' (person_id=2969632)
    'Cristina Liz': None,  # TMDB propuso: 'Cristina Liz Graña' (person_id=2434919)
    'Cristina Menéndez': None,  # TMDB propuso: 'Cristina Menéndez Torralba' (person_id=5181798)
    'Cristóbal Fernández Fernández': None,  # TMDB propuso: 'Cristóbal Fernández' (person_id=1817389)
    'Curro Sánchez Varela': None,  # TMDB propuso: 'Francisco Sánchez Varela' (person_id=1373415)
    'César Souto': None,  # TMDB propuso: 'César Souto Vilanova' (person_id=2434913)
    'Dani Alonso': None,  # TMDB propuso: 'Daniel Alonso Castelo' (person_id=3343684)
    'Darío García García': None,  # TMDB propuso: 'Darío García Sanz' (person_id=5775607)
    'David Martín-Porras': None,  # TMDB propuso: 'David Martín Porras' (person_id=1264790)
    'David Orea Arribas': None,  # TMDB propuso: 'David Orea' (person_id=2747679)
    'David R. Ros': None,  # TMDB propuso: 'D. Ross Lederman' (person_id=111462)
    'Eduardo Sola': None,  # TMDB propuso: 'Eduardo Solá Franco' (person_id=1958700)
    'Elena Cid': None,  # TMDB propuso: 'Elena Cid Sebastián' (person_id=2297013)
    'Eligio Montero': None,  # TMDB propuso: 'Eligio R. Montero' (person_id=1843397)
    'Eloy Calvo Campos': None,  # TMDB propuso: 'Eloy Calvo' (person_id=2839650)
    'Emanuele Giusto': None,  # TMDB propuso: 'Emanuele Giusto Kantfish' (person_id=4980450)
    'Enrique Iznaola': None,  # TMDB propuso: 'Enrique Iznaola Gómez' (person_id=4580247)
    'Enrique Palacio': None,  # TMDB propuso: 'Carlos Enrique Palacio' (person_id=5307851)
    'Eric Navarro Serna': None,  # TMDB propuso: 'Eric Navarro' (person_id=1302760)
    'Esther Díaz': None,  # TMDB propuso: 'Esther Díaz Rey' (person_id=6389807)
    'Fernando Fernán-Gómez': None,  # TMDB propuso: 'Fernando Fernán Gómez' (person_id=965)
    'Francesca Catalá': None,  # TMDB propuso: 'Francesca Català Margarit' (person_id=2631444)
    'Francisco Vaquero': None,  # TMDB propuso: 'Francisco Vaquero Robustillo' (person_id=4056738)
    'Fèlix Colomer': None,  # TMDB propuso: 'Fèlix Colomer Vallès' (person_id=1835536)
    'Gaspar Broullón': None,  # TMDB propuso: 'Gaspar Broullón Pastoriza' (person_id=5046087)
    'Gon': None,  # TMDB propuso: 'Melo Gon' (person_id=3702941)
    'Gonzalo García Pelayo': None,  # TMDB propuso: 'Gonzalo García-Pelayo' (person_id=1152112)
    'Gonzalo Perdomo - Tafur': None,  # TMDB propuso: 'Gonzalo Perdomo-Tafur' (person_id=5052862)
    'Guillermo Guerrero Conget': None,  # TMDB propuso: 'Guillermo Guerrero' (person_id=1608135)
    'Gustau Hernández': None,  # TMDB propuso: 'Gustau Hernández Mor' (person_id=1361408)
    'Ibán Toledo': None,  # TMDB propuso: 'Iban Toledo Ibañez' (person_id=2218256)
    'Ignacio F Iquino': None,  # TMDB propuso: 'Ignacio F. Iquino' (person_id=53943)
    'Ignacio Ortega': None,  # TMDB propuso: 'Ignacio Ortega Campos' (person_id=4580248)
    'Imanol Ortiz': None,  # TMDB propuso: 'Imanol Ortiz López' (person_id=1804069)
    'Inge Mendioroz Ibáñez': None,  # TMDB propuso: 'Inge Mendioroz' (person_id=2468094)
    'Irati Gorostidi Agirretxe': None,  # TMDB propuso: 'Irati Gorostidi' (person_id=1708407)
    'Irene Gutiérrez': None,  # TMDB propuso: 'Irene Gutiérrez Caba' (person_id=548748)
    'Isabel Peña Domingo': None,  # TMDB propuso: 'Isabel Peña' (person_id=1303404)
    'Isaías Cruz': None,  # TMDB propuso: 'Isaías Cruz Irujo' (person_id=5338995)
    'Ivy Hes': None,  # TMDB propuso: 'Ivy Hesh' (person_id=5086132)
    "J. J. García Galocha 'Galo'": None,  # TMDB propuso: 'Juan Jesús García Galocha' (person_id=2044869)
    'J. L. Rojas': None,  # TMDB propuso: 'J.L. Rojas' (person_id=5252601)
    'J.A Bayona': None,  # TMDB propuso: 'J. A. Bayona' (person_id=51894)
    'Jaime Comas': None,  # TMDB propuso: 'Jaime Comas Gil' (person_id=16320)
    'Jaione Camborda Coll': None,  # TMDB propuso: 'Jaione Camborda' (person_id=1836299)
    'Javier Barreira': None,  # TMDB propuso: 'Javier López Barreira' (person_id=1133344)
    'Javier Martín Martín': None,  # TMDB propuso: 'Javier Martín' (person_id=576087)
    'Javier Martínez': None,  # TMDB propuso: 'Javier Sánchez Martínez' (person_id=5348288)
    'Jerónimo Silvio Iglesias': None,  # TMDB propuso: 'Jerónimo Silvio Iglesias Aparicio' (person_id=3511155)
    'Jone Ibarretxe': None,  # TMDB propuso: 'Jone Ibarretxe de la Cal' (person_id=4919663)
    'Jordi Morató': None,  # TMDB propuso: 'Jordi Morató Pujol' (person_id=5729075)
    'Jorge Acebo': None,  # TMDB propuso: 'Jorge Acebo Canedo' (person_id=2570550)
    'Jorge Peña': None,  # TMDB propuso: 'Jorge Peña Pezo' (person_id=5834871)
    'Jose Garci': None,  # TMDB propuso: 'José Luis Garci' (person_id=103511)
    'Jose Luis Luis Font': None,  # TMDB propuso: 'José Luis Font' (person_id=143847)
    'Jose Maria Rodriguez': None,  # TMDB propuso: 'Maria Jose Pacreu Rodriguez' (person_id=5322138)
    'Josema García-Pelayo': None,  # TMDB propuso: 'Josema García Pelayo' (person_id=3511158)
    'Josep T. París': None,  # TMDB propuso: 'Josep T.París' (person_id=5377723)
    'Joserra Halcón': None,  # TMDB propuso: 'Joserra Halcón Bejarano' (person_id=1809334)
    'José A. Torres': None,  # TMDB propuso: 'Antonio Torres Martinó' (person_id=1593605)
    'José Juan Bigas Luna': None,  # TMDB propuso: 'Bigas Luna' (person_id=37577)
    'José Ramón Fernández': None,  # TMDB propuso: 'Jose Ramón Fernández Pinazo' (person_id=4748222)
    'José Sánchez Montes': None,  # TMDB propuso: 'José Sánchez-Montes' (person_id=1065244)
    'Juan Antonio Moreno': None,  # TMDB propuso: 'Juan Antonio Moreno Amador' (person_id=1463084)
    "Juan Jesús García 'Galo'": None,  # TMDB propuso: 'Juan Jesús García Galocha' (person_id=2044869)
    'Juan Prosper': None,  # TMDB propuso: 'Juan Clemente Prosper' (person_id=1564500)
    'Juan Pérez-Fajardo': None,  # TMDB propuso: 'Juan Perez Fajardo' (person_id=3960733)
    'Juanfran Jacinto': None,  # TMDB propuso: 'Juan Fran Jacinto' (person_id=2385196)
    'Juanjo Giménez': None,  # TMDB propuso: 'Juanjo Giménez Peña' (person_id=1459948)
    'Juanma Betancort': None,  # TMDB propuso: 'Juanma Villar Betancort' (person_id=1866952)
    'Julio C. Sánchez': None,  # TMDB propuso: 'Julio César Sánchez' (person_id=2427245)
    'Julio De La Fuente': None,  # TMDB propuso: 'Julio S. de la Fuente' (person_id=4865414)
    'Leone Sergio': None,  # TMDB propuso: 'Sergio Leone' (person_id=4385)
    'Li Jianping': None,  # TMDB propuso: 'Jianping Li' (person_id=3000259)
    'Liena Cid': None,  # TMDB propuso: 'Liena Cid Navia' (person_id=5318950)
    'Lisanda López Fabé': None,  # TMDB propuso: 'Lisandra López Fabe' (person_id=2799326)
    'Lluís Miñarro': None,  # TMDB propuso: 'Luis Miñarro' (person_id=28503)
    'Lola Maldonado Salvador': None,  # TMDB propuso: 'Lola Salvador' (person_id=1871730)
    'Lola Salvador Maldonado': None,  # TMDB propuso: 'Lola Salvador' (person_id=1871730)
    'Louise Andersen': None,  # TMDB propuso: 'Louise Brix Andersen' (person_id=5228746)
    'Lucía Casañ': None,  # TMDB propuso: 'Lucía Casañ Rodríguez' (person_id=4776926)
    'Lucía Herrera Pérez': None,  # TMDB propuso: 'Lucía Herrera' (person_id=3374272)
    'Luis Acebes': None,  # TMDB propuso: 'Luis Acebes Navarro' (person_id=2710782)
    'Luis Alegre': None,  # TMDB propuso: 'Luis Alegre Saz' (person_id=1074041)
    'Luis Gabriel Beristáin': None,  # TMDB propuso: 'Gabriel Beristain' (person_id=10832)
    'Luis Garcia-Berlanga': None,  # TMDB propuso: 'Luis García Berlanga' (person_id=37493)
    'Luis Muñoz Cubillo': None,  # TMDB propuso: 'Luis (Soto) Muñoz' (person_id=3829133)
    'Luis Peñafiel': None,  # TMDB propuso: 'Chicho Ibáñez Serrador' (person_id=133004)
    'Luis Sánchez-Polack': None,  # TMDB propuso: 'Luis Sánchez Polack' (person_id=232130)
    'Luisa García Grajalva': None,  # TMDB propuso: 'Luisa García-Grajalva' (person_id=3564092)
    'Luisa Grajalva': None,  # TMDB propuso: 'Luisa García-Grajalva' (person_id=3564092)
    'Manuel Burque Hodgson': None,  # TMDB propuso: 'Manuel Burque' (person_id=1434231)
    'Manuel Cabo': None,  # TMDB propuso: 'Manuel Cabo Sánchez' (person_id=5331869)
    'Manuel Martínez Velasco': None,  # TMDB propuso: 'Manuel M. Velasco' (person_id=1195190)
    'Mar De Dios Rodríguez': None,  # TMDB propuso: 'María del Mar de Dios Rodríguez' (person_id=5122410)
    'Marc Sempere Moya': None,  # TMDB propuso: 'Marc Sempere-Moya' (person_id=3268460)
    'Margarita Ledo Andión': None,  # TMDB propuso: 'Margarita Ledo' (person_id=2348612)
    'Mariano Ozores': None,  # TMDB propuso: 'Mariano Ozores Puchol' (person_id=229765)
    'Mariano Rojo': None,  # TMDB propuso: 'Mariano Alejandro Rojo' (person_id=4475946)
    'Marta-Libertad': None,  # TMDB propuso: 'Marta-Libertad Castillo' (person_id=2654024)
    'Michael White': None,  # TMDB propuso: 'Michael Jai White' (person_id=64856)
    'Miguel Angel Trujillo': None,  # TMDB propuso: 'Miguel Angel Bonequi Trujillo' (person_id=2927784)
    'Miguel Herrero Herrero': None,  # TMDB propuso: 'Miguel Herrero' (person_id=4951153)
    'Miguel Mejías': None,  # TMDB propuso: 'Miguel A. Mejias' (person_id=2760432)
    'Miguel Ángel Barroso': None,  # TMDB propuso: 'Miguel Angel Barroso Garcia' (person_id=2764547)
    'Miguel Ángel Vivas Moreno': None,  # TMDB propuso: 'Miguel Ángel Vivas' (person_id=140396)
    'Mikel Mas': None,  # TMDB propuso: 'Mikel Mas Bilbao' (person_id=3212690)
    'Monso': None,  # TMDB propuso: 'Percy Monso' (person_id=6371154)
    'Mónica Cambra Domínguez': None,  # TMDB propuso: 'Mònica Cambra' (person_id=2661260)
    'Nicolás Muñoz Avia': None,  # TMDB propuso: 'Nicolás Muñoz' (person_id=260953)
    'Nino Fontán': None,  # TMDB propuso: 'Nino Fontán Allen' (person_id=4919676)
    'Pablo Hernando Esquisabel': None,  # TMDB propuso: 'Pablo Hernando' (person_id=1185975)
    'Pau Canivell Gámez': None,  # TMDB propuso: 'Pau Canivell' (person_id=3304976)
    'Pau García Pérez De Lara': None,  # TMDB propuso: 'Pablo García Pérez de Lara' (person_id=1013025)
    'Paul Urkijo': None,  # TMDB propuso: 'Paul Urkijo Alijo' (person_id=1428818)
    'Pawel Pawlikowski': None,  # TMDB propuso: 'Paweł Pawlikowski' (person_id=64194)
    'Pedro Martín-Calero': None,  # TMDB propuso: 'Pedro Martín Calero' (person_id=1694893)
    'Pedro Pérez-Rosado': None,  # TMDB propuso: 'Pedro Pérez Rosado' (person_id=1036769)
    'Pello Gutiérrez Peñalba': None,  # TMDB propuso: 'Pello Gutiérrez' (person_id=2261544)
    'Phil Ivanusic-Vallée': None,  # TMDB propuso: 'Phil Ivanusic' (person_id=1246601)
    'Rafael Calatayud Cano': None,  # TMDB propuso: 'Rafa Calatayud Cano' (person_id=4223120)
    'Rafael G. Sánchez': None,  # TMDB propuso: 'Rafael Sánchez G.' (person_id=4603658)
    'Rafael J Salvia': None,  # TMDB propuso: 'Rafael J. Salvia' (person_id=98499)
    'Rafael Torrecilla': None,  # TMDB propuso: 'Rafael María Torrecilla' (person_id=1269221)
    'Ray(Jorge Loriga) Loriga': None,  # TMDB propuso: 'Ray Loriga' (person_id=3790)
    'Samuel Martín Delgado': None,  # TMDB propuso: 'Samuel M. Delgado' (person_id=1539731)
    'Santos Blanco': None,  # TMDB propuso: 'Blanco Santos' (person_id=1828428)
    'Sean Patrick O´Reilly': None,  # TMDB propuso: "Sean Patrick O'Reilly" (person_id=131416)
    'Sergio Montero': None,  # TMDB propuso: 'Sergio Montero Fernández' (person_id=5070921)
    'Sergio Rodrigo': None,  # TMDB propuso: 'Sergio Rodrigo Ruiz' (person_id=5503054)
    'Stojan Ðordevic': None,  # TMDB propuso: 'Stojan Đorđević' (person_id=1593464)
    'Tabatta Salinas': None,  # TMDB propuso: 'Tabatta Salinas Caballero' (person_id=4206690)
    'Tomás Aceituno': None,  # TMDB propuso: 'Tomás Aceituno Maldonado' (person_id=1458492)
    'Venci D. Kostov': None,  # TMDB propuso: 'Venci Kostov' (person_id=2192826)
    'Víctor Alonso Berbel': None,  # TMDB propuso: 'Victor Alonso-Berbel' (person_id=2334053)
    'Wang Xianping': None,  # TMDB propuso: 'Xianping Wang' (person_id=2266640)
    'Xabier Mina': None,  # TMDB propuso: 'Xabier Mina Ederra' (person_id=5338994)
    'Xavier Esteban': None,  # TMDB propuso: 'Xavier Esteban Casas' (person_id=3947595)
    'Xavier Torres Lliteras': None,  # TMDB propuso: 'Xavi Torres' (person_id=4034117)
    'Álex O´Dogherty': None,  # TMDB propuso: "Álex O'Dogherty" (person_id=1061514)
    'Álvaro Fernández - Armero': None,  # TMDB propuso: 'Álvaro Fernández Armero' (person_id=563824)
    'Álvaro Lión-Depetre': None,  # TMDB propuso: 'Álvaro Lion Depetre' (person_id=2014388)
    'Ángel Alonso': None,  # TMDB propuso: 'Miguel Ángel Alonso' (person_id=5115816)
    'Ángel Puado': None,  # TMDB propuso: 'Ángel Puado Veloso' (person_id=2864402)
    'Ángela Gallardo': None,  # TMDB propuso: 'Ángela Gallardo Bernal' (person_id=1649980)
    'Ángeles González-Sinde Reig': None,  # TMDB propuso: 'Ángeles González-Sinde' (person_id=230209)
}

In [22]:
for nombre, person_id_correcto in OVERRIDES_PERSON_SEARCH.items():
    if nombre not in person_search_cache:
        continue
    if person_id_correcto is None:
        person_search_cache[nombre] = {"person_id": None, "nombre_tmdb": None}
    else:
        person_search_cache[nombre]["person_id"] = person_id_correcto
        person_search_cache[nombre]["nombre_tmdb"] = None  # se pierde la referencia del nombre TMDB viejo, no importa

guardar_cache_json(PERSON_SEARCH_CACHE, person_search_cache)
print(f"✓ {len([v for v in OVERRIDES_PERSON_SEARCH.values() if v is not None])} overrides aplicados a la caché")

✓ 1 overrides aplicados a la caché


## 7. Enriquecimiento de persona -- fuente 1: TMDB (sexo + país)

In [23]:
def obtener_persona_tmdb(person_id, session, timeout=15):
    url = f"https://api.themoviedb.org/3/person/{person_id}"
    params = {"api_key": TMDB_API_KEY, "language": "es-ES"}
    try:
        r = session.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        return {
            "person_id": person_id,
            "nombre": data.get("name"),
            "sexo": GENERO_MAP.get(data.get("gender"), None),
            "place_of_birth": data.get("place_of_birth"),
        }
    except Exception as e:
        print(f"  Error persona person_id={person_id}: {e}")
        return {"person_id": person_id, "nombre": None, "sexo": None, "place_of_birth": None}


def extraer_pais_aprox(place_of_birth):
    if not place_of_birth or not isinstance(place_of_birth, str):
        return None, "sin_dato"
    partes = [p.strip() for p in place_of_birth.split(",") if p.strip()]
    if not partes:
        return None, "sin_dato"
    return partes[-1], "heuristico"

print("✓ Funciones de enriquecimiento de persona vía TMDB listas (reutilizadas)")


✓ Funciones de enriquecimiento de persona vía TMDB listas (reutilizadas)


In [24]:
PERSONAS_CACHE = CSV_DIR / "personas_cache.csv"

if PERSONAS_CACHE.exists():
    personas_df = pd.read_csv(PERSONAS_CACHE, sep=';')
    persons_hechos = set(personas_df['person_id'].dropna().unique())
    print(f"Reanudando: {len(persons_hechos)} personas ya procesadas")
else:
    personas_df = pd.DataFrame()
    persons_hechos = set()
    print("Empezando desde cero")

person_ids_pendientes = [
    pid for pid in credits_df['person_id'].dropna().unique()
    if pid not in persons_hechos
]
print(f"Pendientes: {len(person_ids_pendientes)} personas únicas")

registros_p = []
for i, person_id in enumerate(person_ids_pendientes, 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(person_ids_pendientes)}] person_id={person_id}")

    info = obtener_persona_tmdb(person_id, session)
    pais_aprox, confianza = extraer_pais_aprox(info["place_of_birth"])
    info["pais_aprox"] = pais_aprox
    info["confianza_pais"] = confianza
    registros_p.append(info)

    if i % 100 == 0:
        batch = pd.DataFrame(registros_p)
        personas_df = pd.concat([personas_df, batch], ignore_index=True)
        personas_df.to_csv(PERSONAS_CACHE, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
        registros_p = []
        print("  ✓ Guardado parcial")

    time.sleep(0.25)

if registros_p:
    batch = pd.DataFrame(registros_p)
    personas_df = pd.concat([personas_df, batch], ignore_index=True)
    personas_df.to_csv(PERSONAS_CACHE, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)

print(f"\n✓ Personas completadas: {len(personas_df)}")
print(f"Con sexo declarado: {personas_df['sexo'].notna().sum()} / {len(personas_df)}")
print(f"Con país (heurístico): {personas_df['pais_aprox'].notna().sum()} / {len(personas_df)}")


Empezando desde cero
Pendientes: 2262 personas únicas
  [25/2262] person_id=2474255.0
  [50/2262] person_id=4237979.0
  [75/2262] person_id=3998718.0
  [100/2262] person_id=1360055.0
  ✓ Guardado parcial
  [125/2262] person_id=3959289.0
  [150/2262] person_id=2391063.0
  [175/2262] person_id=5361958.0
  [200/2262] person_id=1170687.0
  ✓ Guardado parcial
  [225/2262] person_id=589553.0
  [250/2262] person_id=1428818.0
  [275/2262] person_id=19841.0
  [300/2262] person_id=1051295.0
  ✓ Guardado parcial
  [325/2262] person_id=1838478.0
  [350/2262] person_id=2927784.0
  [375/2262] person_id=5247630.0
  [400/2262] person_id=2558765.0
  ✓ Guardado parcial
  [425/2262] person_id=2853009.0
  [450/2262] person_id=2998049.0
  [475/2262] person_id=2263746.0
  [500/2262] person_id=2545879.0
  ✓ Guardado parcial
  [525/2262] person_id=3405709.0
  [550/2262] person_id=132450.0
  [575/2262] person_id=1302077.0
  [600/2262] person_id=2046902.0
  ✓ Guardado parcial
  [625/2262] person_id=5842166.0
  

## 8. Enriquecimiento de persona -- fuente 2: Wikidata (huecos de TMDB)

Se usa el `imdb_id` que TMDB da vía `external_ids` como clave de búsqueda
en Wikidata (propiedad P345). Solo aplica a personas que TMDB sí encontró
(si TMDB no la encontró en absoluto, no hay `imdb_id` disponible por
ninguna vía -- ver aviso en la siguiente sección).

In [25]:
def obtener_imdb_id_tmdb(person_id, session, timeout=15):
    url = f"https://api.themoviedb.org/3/person/{person_id}"
    params = {"api_key": TMDB_API_KEY, "language": "es-ES", "append_to_response": "external_ids"}
    try:
        r = session.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r.json().get("external_ids", {}).get("imdb_id")
    except Exception as e:
        print(f"  Error external_ids person_id={person_id}: {e}")
        return None

print("✓ obtener_imdb_id_tmdb definida (reutilizada)")


✓ obtener_imdb_id_tmdb definida (reutilizada)


In [26]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"


def consultar_wikidata_persona(imdb_id, timeout=30, reintentos=1):
    query = f"""
    SELECT ?persona ?sexoLabel ?paisLabel WHERE {{
      ?persona wdt:P345 "{imdb_id}" .
      OPTIONAL {{ ?persona wdt:P21 ?sexo . }}
      OPTIONAL {{ ?persona wdt:P27 ?pais . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es". }}
    }}
    LIMIT 1
    """
    headers = {"User-Agent": "ProyectoAcademicoLatam/1.0"}
    for intento in range(reintentos + 1):
        try:
            r = requests.get(WIKIDATA_SPARQL, params={"query": query, "format": "json"},
                              headers=headers, timeout=timeout)
            r.raise_for_status()
            bindings = r.json()["results"]["bindings"]
            if not bindings:
                return None, None, None
            b = bindings[0]
            sexo = b.get("sexoLabel", {}).get("value")
            pais = b.get("paisLabel", {}).get("value")
            persona_url = b.get("persona", {}).get("value")
            qid = persona_url.rsplit("/", 1)[-1] if persona_url else None
            return sexo, pais, qid
        except Exception as e:
            if intento < reintentos:
                time.sleep(2)
                continue
            print(f"  Error Wikidata imdb_id={imdb_id}: {e}")
            return None, None, None

print("✓ consultar_wikidata_persona definida (reutilizada)")


✓ consultar_wikidata_persona definida (reutilizada)


In [27]:
faltantes = personas_df[personas_df['sexo'].isna() | personas_df['pais_aprox'].isna()].copy()
print(f"Personas con algún dato faltante tras TMDB: {len(faltantes)} / {len(personas_df)}")

IMDB_ID_CACHE = CSV_DIR / "imdb_id_por_persona_cache.json"
imdb_id_cache = cargar_cache_json(IMDB_ID_CACHE)

for i, person_id in enumerate(faltantes['person_id'], 1):
    key = str(int(person_id))
    if key in imdb_id_cache:
        continue
    if i % 25 == 0:
        print(f"  [{i}/{len(faltantes)}] person_id={person_id}")
    imdb_id_cache[key] = obtener_imdb_id_tmdb(person_id, session)
    if i % 25 == 0:
        guardar_cache_json(IMDB_ID_CACHE, imdb_id_cache)
    time.sleep(0.25)

guardar_cache_json(IMDB_ID_CACHE, imdb_id_cache)
faltantes['imdb_id'] = faltantes['person_id'].map(lambda pid: imdb_id_cache.get(str(int(pid))))
print(f"\nCon imdb_id encontrado: {faltantes['imdb_id'].notna().sum()} / {len(faltantes)}")


Personas con algún dato faltante tras TMDB: 1706 / 2262
  [25/1706] person_id=2445361.0
  [50/1706] person_id=3998718.0
  [75/1706] person_id=3438202.0
  [100/1706] person_id=1756181.0
  [125/1706] person_id=2632890.0
  [150/1706] person_id=2225317.0
  [175/1706] person_id=1506375.0
  [200/1706] person_id=1950950.0
  [225/1706] person_id=1507365.0
  [250/1706] person_id=1304466.0
  [275/1706] person_id=1133344.0
  [300/1706] person_id=4762461.0
  [325/1706] person_id=3592180.0
  [350/1706] person_id=2545879.0
  [375/1706] person_id=2556742.0
  [400/1706] person_id=2204600.0
  [425/1706] person_id=2046902.0
  [450/1706] person_id=82726.0
  [475/1706] person_id=4796018.0
  [500/1706] person_id=5295734.0
  [525/1706] person_id=2806172.0
  [550/1706] person_id=5959830.0
  [575/1706] person_id=1368562.0
  [600/1706] person_id=1544003.0
  [625/1706] person_id=2873841.0
  [650/1706] person_id=1264790.0
  [675/1706] person_id=59144.0
  [700/1706] person_id=564076.0
  [725/1706] person_id=53844

In [28]:
WIKIDATA_PERSONAS_CACHE = CSV_DIR / "wikidata_personas_cache.json"
wd_personas_cache = cargar_cache_json(WIKIDATA_PERSONAS_CACHE)
print(f"Caché existente: {len(wd_personas_cache)} personas")

con_imdb = faltantes[faltantes['imdb_id'].notna()]
pendientes_wd_personas = [
    fila for fila in con_imdb.itertuples() if fila.imdb_id not in wd_personas_cache
]
print(f"Pendientes: {len(pendientes_wd_personas)} / {len(con_imdb)}")

for i, fila in enumerate(pendientes_wd_personas, 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(pendientes_wd_personas)}] {fila.imdb_id}")
    sexo_wd, pais_wd, qid_wd = consultar_wikidata_persona(fila.imdb_id)
    wd_personas_cache[fila.imdb_id] = {
        "person_id": fila.person_id, "sexo_wikidata": sexo_wd,
        "pais_wikidata": pais_wd, "wikidata_qid": qid_wd,
    }
    if i % 25 == 0:
        guardar_cache_json(WIKIDATA_PERSONAS_CACHE, wd_personas_cache)
        print("  ✓ Guardado parcial")
    time.sleep(0.3)

guardar_cache_json(WIKIDATA_PERSONAS_CACHE, wd_personas_cache)

wd_df = pd.DataFrame([
    {"person_id": v["person_id"], "sexo_wikidata": v["sexo_wikidata"],
     "pais_wikidata": v["pais_wikidata"], "wikidata_qid": v["wikidata_qid"]}
    for v in wd_personas_cache.values()
])
print(f"\n✓ Total en caché: {len(wd_df)}")
print(f"Con sexo desde Wikidata: {wd_df['sexo_wikidata'].notna().sum()}")
print(f"Con país desde Wikidata: {wd_df['pais_wikidata'].notna().sum()}")


Caché existente: 0 personas
Pendientes: 516 / 516
  [25/516] nm4652014
  ✓ Guardado parcial
  [50/516] nm2516950
  ✓ Guardado parcial
  [75/516] nm1141070
  ✓ Guardado parcial
  [100/516] nm4904829
  ✓ Guardado parcial
  [125/516] nm4698030
  ✓ Guardado parcial
  [150/516] nm2856376
  ✓ Guardado parcial
  [175/516] nm0293139
  ✓ Guardado parcial
  [200/516] nm0753807
  ✓ Guardado parcial
  [225/516] nm1396295
  ✓ Guardado parcial
  [250/516] nm0554693
  ✓ Guardado parcial
  [275/516] nm0765205
  ✓ Guardado parcial
  [300/516] nm5327902
  ✓ Guardado parcial
  [325/516] nm1892876
  ✓ Guardado parcial
  [350/516] nm1779829
  ✓ Guardado parcial
  [375/516] nm1147094
  ✓ Guardado parcial
  [400/516] nm1503330
  ✓ Guardado parcial
  [425/516] nm1281606
  ✓ Guardado parcial
  [450/516] nm2246351
  ✓ Guardado parcial
  [475/516] nm0753849
  ✓ Guardado parcial
  [500/516] nm0019115
  ✓ Guardado parcial

✓ Total en caché: 516
Con sexo desde Wikidata: 341
Con país desde Wikidata: 263


In [29]:
personas_df = personas_df.merge(wd_df, on='person_id', how='left')

MAPA_SEXO_WD = {"masculino": "hombre", "femenino": "mujer", "hombre": "hombre",
                 "mujer": "mujer", "no binario": "no_binario"}
personas_df['sexo_wikidata_norm'] = personas_df['sexo_wikidata'].str.lower().map(MAPA_SEXO_WD)

personas_df['sexo_final'] = personas_df['sexo'].fillna(personas_df['sexo_wikidata_norm'])
personas_df['fuente_sexo'] = personas_df['sexo'].notna().map({True: 'tmdb', False: None})
personas_df.loc[personas_df['sexo'].isna() & personas_df['sexo_wikidata_norm'].notna(), 'fuente_sexo'] = 'wikidata'

print(f"Cobertura sexo tras TMDB + Wikidata: {personas_df['sexo_final'].notna().sum()} / {len(personas_df)}")


Cobertura sexo tras TMDB + Wikidata: 1042 / 2262


## 9. Enriquecimiento de persona -- fuente 3: IMDb directo (solo país)

Para quien tenga `imdb_id` pero Wikidata tampoco tuviera el país. Se visita
la ficha de la persona en IMDb: primero se intenta el campo estructurado
`birthLocation`, y si no está, se lee la biografía en prosa (patrón
"Nació [<fecha>] en <Ciudad>, <País>").

**No se usa para sexo** -- IMDb no tiene un campo estructurado equivalente
al `gender` de TMDB, y la inferencia por pronombres en la biografía
resultó poco fiable en las pruebas.

In [30]:
def extraer_pais_de_bio_imdb(texto_bio):
    if not texto_bio:
        return None
    m = re.search(
        r"[Nn]aci[oó](?:.*?)\ben\s+[^,]+,\s*([A-ZÁÉÍÓÚÑ][\wÀ-ÿ]*(?:\s+[A-ZÁÉÍÓÚÑ][\wÀ-ÿ]*)*)",
        texto_bio
    )
    return m.group(1).strip() if m else None


def obtener_pais_imdb(imdb_id, driver):
    """
    1. birthLocation estructurado (regex sobre el HTML, el JSON embebido es
       demasiado grande/inestable para parsear entero de forma fiable)
    2. si no está, biografía en prosa (data-testid="bio-content")
    """
    try:
        driver.get(f"https://www.imdb.com/es/name/{imdb_id}/")
    except Exception as e:
        print(f"  Error cargando IMDb {imdb_id}: {e}")
        return None, None

    html = driver.page_source

    m = re.search(r'"birthLocation":\{"text":"([^"]+)"', html)
    if m:
        lugar = m.group(1)
        partes = [p.strip() for p in lugar.split(",") if p.strip()]
        if partes:
            return partes[-1], "imdb_estructurado"

    soup = BeautifulSoup(html, "html.parser")
    bio_div = soup.find(attrs={"data-testid": "bio-content"})
    texto_bio = bio_div.get_text(" ", strip=True) if bio_div else None
    pais_bio = extraer_pais_de_bio_imdb(texto_bio)
    if pais_bio:
        return pais_bio, "imdb_bio_texto"

    return None, None

print("✓ Funciones de fallback a IMDb definidas")


✓ Funciones de fallback a IMDb definidas


In [31]:
faltantes_pais_imdb = faltantes[
    faltantes['imdb_id'].notna()
    & faltantes['person_id'].isin(
        personas_df.loc[personas_df['pais_wikidata'].isna(), 'person_id']
    )
].copy()
print(f"Personas a intentar en IMDb (con imdb_id, sin país tras TMDB+Wikidata): {len(faltantes_pais_imdb)}")

IMDB_PAIS_CACHE = CSV_DIR / "imdb_pais_cache.json"
imdb_pais_cache = cargar_cache_json(IMDB_PAIS_CACHE)

driver = iniciar_driver()
try:
    for i, fila in enumerate(faltantes_pais_imdb.itertuples(), 1):
        if fila.imdb_id in imdb_pais_cache:
            continue
        if i % 20 == 0:
            print(f"  [{i}/{len(faltantes_pais_imdb)}] {fila.imdb_id}")
        pais, fuente = obtener_pais_imdb(fila.imdb_id, driver)
        imdb_pais_cache[fila.imdb_id] = {"person_id": fila.person_id, "pais": pais, "fuente": fuente}
        if i % 20 == 0:
            guardar_cache_json(IMDB_PAIS_CACHE, imdb_pais_cache)
            print("  ✓ Guardado parcial")
        time.sleep(random.uniform(2.0, 3.5))
finally:
    driver.quit()
    print("✓ Driver cerrado")

guardar_cache_json(IMDB_PAIS_CACHE, imdb_pais_cache)

imdb_pais_df = pd.DataFrame([
    {"person_id": v["person_id"], "pais_imdb": v["pais"], "fuente_pais_imdb": v["fuente"]}
    for v in imdb_pais_cache.values()
])
print(f"\n✓ Con país desde IMDb: {imdb_pais_df['pais_imdb'].notna().sum()} / {len(imdb_pais_df)}")


Personas a intentar en IMDb (con imdb_id, sin país tras TMDB+Wikidata): 253
  [20/253] nm5648314
  ✓ Guardado parcial
  [40/253] nm3324609
  ✓ Guardado parcial
  [60/253] nm3911679
  ✓ Guardado parcial
  [80/253] nm0169180
  ✓ Guardado parcial
  [100/253] nm0306443
  ✓ Guardado parcial
  [120/253] nm1511424
  ✓ Guardado parcial
  [140/253] nm2609612
  ✓ Guardado parcial
  [160/253] nm4559927
  ✓ Guardado parcial
  [180/253] nm3590688
  ✓ Guardado parcial
  [200/253] nm2330982
  ✓ Guardado parcial
  [220/253] nm5595047
  ✓ Guardado parcial
  [240/253] nm0515848
  ✓ Guardado parcial
✓ Driver cerrado

✓ Con país desde IMDb: 0 / 253


In [34]:
faltantes_pais_imdb.head()

,person_id,nombre,sexo,place_of_birth,pais_aprox,confianza_pais,imdb_id
2,1114075.0,Vicente Pérez Herrero,NaN,NaN,NaN,sin_dato,nm0701942
16,1284828.0,Juan Barrero,NaN,NaN,NaN,sin_dato,nm2209314
23,131587.0,Víctor Monigote,NaN,NaN,NaN,sin_dato,nm2857166
32,1105225.0,Héctor Claramunt,NaN,NaN,NaN,sin_dato,nm0163467
39,1063543.0,Verónica Fernández,mujer,NaN,NaN,sin_dato,nm0273784


In [35]:
driver = iniciar_driver()
try:
    imdb_id_prueba = "nm5648314"  # uno de los que falló arriba
    driver.get(f"https://www.imdb.com/es/name/{imdb_id_prueba}/")
    time.sleep(2)  # margen extra, por si acaso
    html = driver.page_source

    print("Longitud del HTML recibido:", len(html))
    print()
    print("¿Aparece 'birthLocation' en algún sitio?:", "birthLocation" in html)
    print("¿Aparece 'bio-content'?:", "bio-content" in html)
    print("¿Aparece 'captcha' o 'blocked' o 'unusual traffic'?:",
          any(p in html.lower() for p in ["captcha", "blocked", "unusual traffic", "access denied"]))
    print()
    print("Título de la página (<title>):")
    m = re.search(r"<title>(.*?)</title>", html)
    print(m.group(1) if m else "no encontrado")
    print()
    print("Primeros 1500 caracteres del HTML:")
    print(html[:1500])
finally:
    driver.quit()

Longitud del HTML recibido: 9530

¿Aparece 'birthLocation' en algún sitio?: False
¿Aparece 'bio-content'?: False
¿Aparece 'captcha' o 'blocked' o 'unusual traffic'?: True

Título de la página (<title>):
Human Verification

Primeros 1500 caracteres del HTML:
<html lang="en"><head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Human Verification</title>
    <style>
        body {
            font-family: "Arial";
        }
    </style>
    <script type="text/javascript">
    window.awsWafCookieDomainList = ['imdb.com'];
    window.gokuProps = {
"key":"AQIDAHjcYu/GjX+QlghicBgQ/7bFaQZ+m5FKCMDnO+vTbNg96AGj9CNjkrHJK/VBRvomxSbuAAAAfjB8BgkqhkiG9w0BBwagbzBtAgEAMGgGCSqGSIb3DQEHATAeBglghkgBZQMEAS4wEQQMdM0kgGZ1rETMLYLkAgEQgDvuiNA1zhRYC/5tDKz4jcOZgCNDyskow3mcoI6luFkkNTywR1Da/3ji9w9EdfQGgLyxgN8TPxrx8XqelA==",
          "iv":"CgAGYzJCYwAAGR8L",
          "context":"qle4B/dU2g76XOjtT6bZISfPPHR8OehF+kBlpZeneEn7zdJUUy0+5KQB7kg/PT2tNzQgQ+W8

### IMDb directo -- descartado (bloqueo anti-bot)

Probado contra datos reales: IMDb devuelve una página de verificación
"Human Verification" (AWS WAF Bot Control) en vez del contenido, incluso
con delays de varios segundos entre peticiones -- 0/253 personas
recuperadas en la corrida de prueba. No es un problema de regex ni de
timing, es un bloqueo activo.

**Se abandona esta fuente**, igual que ya se abandonó el matching de
Wikidata por nombre en la fase anterior del proyecto por su alta tasa de
falsos positivos -- es una limitación metodológica documentada, no un
hueco oculto. El país de estas personas queda como venga de TMDB/Wikidata
(o vacío, si ninguna de las dos lo tuvo).

In [32]:
personas_df = personas_df.merge(imdb_pais_df, on='person_id', how='left')
print("✓ País de IMDb incorporado a personas_df")


✓ País de IMDb incorporado a personas_df


## 10. Homologación de país a código ISO alpha-2

Mismo mapa curado en la fase anterior del proyecto. Se reutiliza tal cual
-- es una tabla de traducción estática, no depende del corpus. Es probable
que el corpus ampliado (~10x personas) revele nombres de país no cubiertos
todavía; el resumen final de abajo muestra cuántos quedan sin mapear.

In [36]:
MAPA_PAISES = {
    'Spain': 'ES', 'España': 'ES', 'Spagna': 'ES', 'Illas Canarias': 'ES', 'Galiza': 'ES',
    'France': 'FR', 'Danmark': 'DK', 'Brazil': 'BR', 'Brasil': 'BR',
    'Mexico': 'MX', 'México': 'MX', 'Peru': 'PE', 'Perú': 'PE',
    'Argentina': 'AR', 'Argentina.': 'AR', 'Chile': 'CL', 'Uruguay': 'UY',
    'El Salvador': 'SV', 'Guatemala': 'GT', 'Panama': 'PA', 'Puerto Rico': 'PR',
    'Costa Rica': 'CR', 'Dominican Republic': 'DO', 'Colombia': 'CO', 'Ecuador': 'EC',
    'Bolivia': 'BO', 'Cuba': 'CU', 'Paraguay': 'PY', 'Canada': 'CA', 'USA': 'US',
    'New York': 'US', 'Iran': 'IR', 'Reino de los Países Bajos': 'NL', 'Italia': 'IT',
    'Reino Unido': 'GB', 'Estados Unidos': 'US', 'Suiza': 'CH',

    # -- añadidos tras revisar el corpus ampliado --
    # Italia (variantes de idioma/erratas)
    'Italy': 'IT', 'Itally': 'IT', 'Regno Unito': 'GB', 'Firenze': 'IT',

    # España (regiones, ciudades, comunidades autónomas, variantes de idioma)
    'Euskal Herria': 'ES', 'Basque Country': 'ES', 'Valladolid - Spain': 'ES',
    'Madrid - Spain': 'ES', 'Canary Islands': 'ES', 'Barcelona': 'ES',
    'Catalunya': 'ES', 'Espagne': 'ES', 'Valencia (Spain)': 'ES',
    'Islas Baleares': 'ES', 'Spai': 'ES', 'Navarra': 'ES', 'Madrid': 'ES',

    # Reino Unido (variantes)
    'UK': 'GB', 'England': 'GB', 'London': 'GB',

    # Francia (variantes)
    'Francia': 'FR', 'Belgique': 'BE',

    # Resto de países (nombre en inglés/español/idioma nativo -> ISO)
    'Venezuela': 'VE', 'Germany': 'DE', 'Alemania': 'DE',
    'Portugal': 'PT', 'Denmark': 'DK', 'Reino de Dinamarca': 'DK',
    'Poland': 'PL', 'Polska': 'PL', 'Switzerland': 'CH', 'Australia': 'AU',
    'Sweden': 'SE', 'Hong Kong': 'HK', 'Türkiye': 'TR', 'Turquía': 'TR',
    'Serbia': 'RS', 'Finland': 'FI', 'Hungary': 'HU', 'Lithuania': 'LT',
    'Lituania': 'LT', 'Норвегия': 'NO', 'Ukraine': 'UA', 'Brasília': 'BR',
    'Belgium': 'BE', 'Afghanistan': 'AF', 'Lebanon': 'LB', 'Morocco': 'MA',
    'República Argentina': 'AR', 'Canadá': 'CA', 'Moldavia': 'MD', 'Rusia': 'RU',
    'Japón': 'JP',

    # -- deliberadamente SIN mapear (ambiguos/no son un país actual) --
    # 'USSR': disuelta en 15 países, no se puede asignar un ISO único con confianza
    # 'Yugoslavia': disuelta en varios países actuales (Serbia, Croacia, Bosnia...)
    # 'Kosovo': estatus disputado, sin ISO alpha-2 oficial universal
    # 'EU': no es un país, es la Unión Europea
}

personas_df['pais_aprox_iso'] = personas_df['pais_aprox'].map(MAPA_PAISES)
personas_df['pais_wikidata_iso'] = personas_df['pais_wikidata'].map(MAPA_PAISES)

personas_df['pais_final'] = personas_df['pais_wikidata_iso'].fillna(personas_df['pais_aprox_iso'])

personas_df['fuente_pais'] = None
personas_df.loc[personas_df['pais_aprox_iso'].notna(), 'fuente_pais'] = 'tmdb_heuristico'
personas_df.loc[personas_df['pais_wikidata_iso'].notna(), 'fuente_pais'] = 'wikidata'

print(f"Cobertura final de país (TMDB + Wikidata; IMDb descartado, ver nota anterior): "
      f"{personas_df['pais_final'].notna().sum()} / {len(personas_df)}")
print(f"\nFuente del país:\n{personas_df['fuente_pais'].value_counts(dropna=False)}")

sin_mapear = pd.concat([
    personas_df.loc[personas_df['pais_aprox'].notna() & personas_df['pais_aprox_iso'].isna(), 'pais_aprox'],
    personas_df.loc[personas_df['pais_wikidata'].notna() & personas_df['pais_wikidata_iso'].isna(), 'pais_wikidata'],
]).value_counts()
print(f"\n⚠ Valores de país sin mapear en MAPA_PAISES (añadir si aparecen aquí):")
print(sin_mapear)


Cobertura final de país (TMDB + Wikidata; IMDb descartado, ver nota anterior): 827 / 2262

Fuente del país:
fuente_pais
None               1435
tmdb_heuristico     565
wikidata            262
Name: count, dtype: int64

⚠ Valores de país sin mapear en MAPA_PAISES (añadir si aparecen aquí):
EU            1
Kosovo        1
USSR          1
Yugoslavia    1
Name: count, dtype: int64


## 11. Correcciones manuales (reutilizadas de la fase anterior)

Mismo diccionario que ya tenías, clave `person_id` de TMDB (estable entre
corridas -- no depende del corpus). Se aplica tal cual, sin tener que
volver a buscar a nadie.

In [38]:
"""
Formato: {person_id: {'sexo': ..., 'pais': ...}}
"""
CORRECCIONES_MANUALES_PERSONAS = {
    3517055: {'sexo': 'hombre', 'pais': 'ES'},
    2018522: {'sexo': 'hombre', 'pais': 'ES'},
    3962124: {'sexo': 'hombre', 'pais': 'MX'},
    1694893: {'sexo': 'hombre', 'pais': 'ES'},
    1980582: {'sexo': 'hombre', 'pais': 'UY'},
    3190577: {'sexo': 'hombre', 'pais': 'ES'},
    2628918: {'sexo': 'hombre', 'pais': 'CO'},
    2018520: {'sexo': 'mujer', 'pais': 'ES'},
    2549328: {'sexo': 'mujer', 'pais': 'ES'},
    3213575: {'pais': 'MX'},
    1115313: {'pais': 'ES'},
    1285223: {'pais': 'ES'},
    1469682: {'sexo': 'hombre', 'pais': 'AR'},
    1398043: {'sexo': 'hombre', 'pais': 'ES'},
    1525399: {'sexo': 'hombre', 'pais': 'ES'},
    3989209: {'sexo': 'hombre', 'pais': 'ES'},
    1533451: {'sexo': 'mujer', 'pais': 'NI'},
    5605266: {'sexo': 'mujer', 'pais': 'PE'},
    1091827: {'sexo': 'hombre', 'pais': 'AR'},
    2568321: {'sexo': 'mujer', 'pais': 'DO'},
    1624073: {'sexo': 'hombre', 'pais': 'EC'},
    4206690: {'sexo': 'mujer', 'pais': 'MX'},
    2662941: {'sexo': 'mujer', 'pais': 'ES'},
    1591887: {'sexo': 'hombre', 'pais': 'MX'},
    2839001: {'sexo': 'hombre', 'pais': 'ES'},
    1731588: {'pais': 'AR'},
    2282004: {'sexo': 'hombre', 'pais': 'ES'},
    4191264: {'sexo': 'mujer', 'pais': 'CO'},
    3202577: {'sexo': 'mujer', 'pais': 'AR'},
    1665562: {'sexo': 'mujer', 'pais': 'IT'},
    3044547: {'sexo': 'hombre', 'pais': 'ES'},
    1338870: {'sexo': 'hombre', 'pais': 'CU'},
    1962957: {'pais': 'ES'},
    2051782: {'pais': 'ES'},
    2472549: {'sexo': 'hombre', 'pais': 'ES'},
    5445665: {'sexo': 'mujer', 'pais': 'BO'},
    5052862: {'sexo': 'hombre', 'pais': 'CO'},
    2555139: {'sexo': 'mujer', 'pais': 'AR'},
    1789288: {'sexo': 'hombre', 'pais': 'CH'},
    2944940: {'sexo': 'hombre', 'pais': 'ES'},
    1805421: {'pais': 'ES'},
    2683614: {'sexo': 'mujer', 'pais': 'AR'},
    1184043: {'sexo': 'hombre', 'pais': 'MX'},
    2427035: {'sexo': 'mujer', 'pais': 'CO'},
    3698547: {'pais': 'ES'},
    2466544: {'sexo': 'hombre', 'pais': 'ES'},
    572115: {'pais': 'PR'},
    5549957: {'sexo': 'hombre', 'pais': 'MX'},
    589119: {'sexo': 'hombre', 'pais': 'MX'},
    1443063: {'sexo': 'hombre', 'pais': 'ES'},
    1740764: {'sexo': 'hombre', 'pais': 'AR'},
    3970485: {'sexo': 'mujer', 'pais': 'PE'},
    2475701: {'pais': 'PE'},
    5455347: {'sexo': 'mujer', 'pais': 'ES'},
    3039206: {'sexo': 'mujer', 'pais': 'ES'},
    1263590: {'sexo': 'hombre', 'pais': 'UY'}, 
    1490109: {'sexo': 'hombre', 'pais': 'ES'},
    1605611: {'sexo': 'hombre', 'pais': 'ES'},
    82726:   {'sexo': 'hombre', 'pais': 'ES'},   # Curro Velázquez
    2924495: {'sexo': 'hombre', 'pais': 'ES'},   # Benjamín Herranz
    1569540: {'sexo': 'mujer',  'pais': 'ES'},   # Bárbara Alpuente
    5727167: {'sexo': 'mujer',  'pais': None},   # Carmen López
    5739963: {'sexo': 'hombre', 'pais': 'ES'},   # Antonio Hernández Ruiz
    4223120: {'sexo': 'hombre', 'pais': 'ES'},   # Rafa Calatayud Cano
    3190577: {'sexo': 'hombre', 'pais': 'ES'},   # Jacob Santana
    4958781: {'sexo': 'hombre', 'pais': 'ES'},   # Kike Maíllo
    2654024: {'sexo': 'mujer',  'pais': 'ES'},   # Marta-Libertad Castillo
    1835761: {'sexo': 'hombre', 'pais': None},   # Gustavo Hernández
    1694893: {'sexo': 'hombre', 'pais': 'ES'},   # Pedro Martín Calero
    5455347: {'sexo': 'mujer',  'pais': 'ES'},   # Teresa Trasancos
    1980582: {'sexo': 'hombre', 'pais': 'IT'},   # Juma Fodde
    3418854: {'sexo': 'hombre', 'pais': 'ES'},   # Daniel Cebrián
    4983818: {'sexo': 'hombre', 'pais': None},   # Ángel Gómez
    1263590: {'sexo': 'hombre', 'pais': 'AR'},   # Manuel Facal
    3039206: {'sexo': 'mujer',  'pais': 'ES'},   # Sandra García Nieto
    5838939: {'sexo': 'mujer',  'pais': 'ES'},   # Marta Medina
    1317749: {'sexo': 'hombre', 'pais': 'ES'},   # David Desola
    1605611: {'sexo': 'hombre', 'pais': 'ES'},   # Marçal Cebrian
    2865703: {'sexo': 'mujer',  'pais': 'ES'},   # Ángela Obón
    4808255: {'sexo': 'mujer',  'pais': 'ES'},   # Sabina Méndez
    3962124: {'sexo': 'hombre', 'pais': 'AR'},   # Marco Lagarde
    3517055: {'sexo': 'hombre', 'pais': 'ES'},   # José Corral Llorente
    3893971: {'sexo': 'hombre', 'pais': 'ES'},   # Xabi Zabaleta
    113407:  {'sexo': 'mujer',  'pais': 'ES'},   # Arantxa Cuesta
    2103455: {'sexo': 'hombre', 'pais': 'AR'},   # Rodolfo Palacios
    4889588: {'sexo': 'hombre', 'pais': 'AR'},   # Pablo Agüero
    2480566: {'sexo': 'mujer',  'pais': 'FR'},   # Katell Guillou
    1157587: {'sexo': 'hombre', 'pais': 'ES'},   # David Baute
    4640765: {'sexo': 'mujer',       'pais': 'ES'},   # Yaiza Berrocal
    2428561: {'sexo': 'hombre',      'pais': 'ES'},   # Pau Calpe Rufat
    1876833: {'sexo': 'mujer',       'pais': 'ES'},   # Marta Grau
    3156507: {'sexo': 'hombre',      'pais': 'UY'},   # Rafael Russo
    5020311: {'sexo': 'hombre',      'pais': 'ES'},   # Samuel del Amor
    2018520: {'sexo': 'mujer',       'pais': 'ES'},   # Ángeles Hernández
    6360702: {'sexo': 'hombre',      'pais': 'ES'},   # David Matamoros Manteca
    1631846: {'sexo': 'hombre',      'pais': 'AR'},   # Álvaro Urtizberea
    1970518: {'sexo': 'mujer',       'pais': 'AR'},   # Erika Halvorsen
    2662836: {'sexo': 'no_binario',  'pais': 'ES'},   # Begoña Arostegui
    2555139: {'sexo': 'mujer',       'pais': 'AR'},   # Mariana Barassi
    31328:   {'sexo': 'hombre',      'pais': 'ES'},   # Diego Cagide
    3626927: {'sexo': 'hombre',      'pais': 'AR'},   # Diego Lucero
    4475946: {'sexo': 'hombre',      'pais': 'AR'},   # Mariano Alejandro Rojo
    5052862: {'sexo': 'hombre',      'pais': 'CO'},   # Gonzalo Perdomo-Tafur
    3530478: {'sexo': 'hombre',      'pais': 'ES'},   # Andrés Martorell
    4385976: {'sexo': 'hombre',      'pais': 'ES'},   # Javier Polo
    1608135: {'sexo': 'hombre',      'pais': 'ES'},   # Guillermo Guerrero
    1462879: {'sexo': 'hombre',      'pais': 'ES'},   # Enric Pardo
    5736743: {'sexo': 'hombre',      'pais': None},   # David Pascual
    4701879: {'sexo': 'hombre',      'pais': 'ES'},   # Ignasi Guerrero
    2472549: {'sexo': 'hombre',      'pais': 'ES'},   # Arturo Méndiz
    2522604: {'sexo': 'mujer',       'pais': 'ES'},   # Claudia Reig
    5777600: {'sexo': 'hombre',      'pais': 'ES'},   # Dani Fabra
    4506103: {'sexo': 'hombre',      'pais': 'VE'},   # Andry José Barrientos
    5232889: {'sexo': 'hombre',      'pais': 'ES'},   # Andrés Clemente
    2018522: {'sexo': 'hombre',      'pais': 'ES'},   # David Matamoros
    2661260: {'sexo': 'mujer',       'pais': 'ES'},   # Mònica Cambra
    3960226: {'sexo': 'mujer',       'pais': 'ES'},  # Ariadna Fortuny
    3960227: {'sexo': 'mujer',  'pais': 'ES'},   # Clàudia Garcia de Dios
    3374272: {'sexo': 'mujer',  'pais': None},   # Lucía Herrera
    1962957: {'sexo': 'mujer',  'pais': 'ES'},   # Meritxell Colell Aparicio
    1338870: {'sexo': 'hombre', 'pais': 'CU'},   # Pavel Giroud
    1764976: {'sexo': 'hombre', 'pais': 'AR'},   # Mariano Donoso
    2962112: {'sexo': 'hombre', 'pais': 'AR'},   # Federico Cardone
    2614503: {'sexo': 'mujer',  'pais': 'AR'},   # Mariana Guzzante
    5674493: {'sexo': 'hombre', 'pais': 'ES'},   # Jesús Caballero
    4572706: {'sexo': 'mujer',  'pais': 'ES'},   # Laura Herrero Garvín
    5548470: {'sexo': 'hombre', 'pais': 'PE'},   # Alejandro Andrade Pease
    3928504: {'sexo': 'hombre', 'pais': 'ES'},   # Armando López Muñoz
    2929692: {'sexo': 'hombre', 'pais': 'ES'},   # Jon Viar
    1731588: {'sexo': 'hombre', 'pais': 'AR'},   # Santiago Fillol
    3772682: {'sexo': 'hombre', 'pais': 'AR'},   # Lucas Vermal
    3772681: {'sexo': 'hombre', 'pais': 'AR'},   # Edgardo Dobry
    2282004: {'sexo': 'hombre', 'pais': 'ES'},   # David Berraondo
    131587:  {'sexo': 'hombre', 'pais': 'ES'},   # Víctor Monigote
    4962452: {'sexo': 'hombre', 'pais': 'AR'},   # Pablo Enrique Bossi
    5500234: {'sexo': 'mujer',  'pais': 'ES'},   # Jenifer de la Rosa Martín
    3202577: {'sexo': 'mujer',  'pais': 'AR'},   # María Zanetti
    1665562: {'sexo': 'mujer',  'pais': 'IT'},   # Maria Mauti
    4080092: {'sexo': 'mujer',  'pais': 'ES'},   # Sara Mesa
    3016177: {'sexo': 'mujer',  'pais': None},   # Mercedes Rodrigo
    5371432: {'sexo': 'mujer',  'pais': 'ES'},   # Helena Girón
    1866952: {'sexo': 'hombre', 'pais': 'ES'},   # Juanma Villar Betancort
    4318654: {'sexo': 'hombre', 'pais': 'CU'},   # Erlyn L. Borges
    4102781: {'sexo': 'hombre', 'pais': 'UY'},   # Facundo Ponce de León
    4865414: {'sexo': 'hombre', 'pais': 'ES'},   # Julio S. de la Fuente
    2239321: {'sexo': 'mujer',  'pais': 'AR'},   # Natacha Kucic
    2467723: {'sexo': 'mujer',  'pais': 'ES'},   # Patricia Pérez Fernández
    2568321: {'sexo': 'mujer',  'pais': 'DO'},   # Johanne Gómez Terrero
    2441053: {'sexo': 'mujer',  'pais': 'AR'},   # Mariana Cangas
    3536688: {'sexo': 'hombre', 'pais': 'ES'},   # Miguel García de la Calera
    4075083: {'sexo': 'mujer',  'pais': 'AR'},   # Carolina Alamino
    5456803: {'sexo': 'hombre', 'pais': 'ES'},   # José del Corral
    2422984: {'sexo': 'hombre', 'pais': 'IT'},   # Mauro Colombo
    2079453: {'sexo': 'hombre', 'pais': 'ES'},   # Andrés Garrigó
    4019154: {'sexo': 'hombre', 'pais': 'ES'},   # Josepmaria Anglès
    2079455: {'sexo': 'hombre', 'pais': 'ES'},   # Josemaría Muñoz
    3102068: {'sexo': 'mujer',  'pais': 'ES'},   # Cira Valiño
    4206690: {'sexo': 'mujer',  'pais': 'MX'},   # Tabatta Salinas Caballero
    2662941: {'sexo': 'mujer',  'pais': 'AR'},   # Andrea Gautier
    1091827: {'sexo': 'hombre', 'pais': 'AR'},   # Diego Yaker
    3632587: {'sexo': 'hombre', 'pais': 'ES'},   # José Manuel Rebollo
    1740764: {'sexo': 'hombre', 'pais': 'AR'},   # Luciano Juncos
    3006682: {'sexo': 'hombre', 'pais': 'CL'},   # Renzo Felippa
    1115313: {'sexo': 'hombre', 'pais': 'ES'},   # Javier Espada
    2799326: {'sexo': 'mujer',  'pais': 'CU'},   # Lisandra López Fabe
    1358333: {'sexo': 'hombre', 'pais': 'NL'},   # Arturo Prins
    1756181: {'sexo': 'mujer',  'pais': 'US'},   # Daresha Kyi
    1161761: {'sexo': 'mujer',  'pais': 'US'},   # Catherine Gund
    1262693: {'sexo': 'hombre', 'pais': 'BR'},   # Daniel Augusto
    4180107: {'sexo': 'hombre', 'pais': 'CU'},   # Ernesto Daranas Serrano
    1525399: {'sexo': 'hombre', 'pais': 'BO'},   # Ida Cuéllar
    2645524: {'sexo': 'hombre', 'pais': 'ES'},   # Juan Muñoz-Tébar
    5036140: {'sexo': 'hombre', 'pais': 'ES'},   # Jorge Cebrián
    5209434: {'sexo': 'hombre', 'pais': None},   # José Luis López
    589777:  {'sexo': 'hombre', 'pais': None},   # Oscar Martín
    1533451: {'sexo': 'mujer',  'pais': 'NI'},   # Laura Baumeister
    2227717: {'sexo': 'mujer',  'pais': 'FR'},   # Laure Desmazières
    3424062: {'sexo': 'hombre', 'pais': 'MX'},   # César Tejeda
    1793738: {'sexo': 'mujer',  'pais': 'ES'},    # Carmen García Rodeja
    1696632: {'sexo': 'hombre', 'pais': 'ES'},   # Eliseo Fernández
    5597231: {'sexo': 'hombre', 'pais': None},   # Diego Martínez
    2944940: {'sexo': 'hombre', 'pais': 'ES'},   # Sergio García de Leaniz
    5013620: {'sexo': 'hombre', 'pais': None},   # Vicente Pérez
    1645082: {'sexo': 'mujer',  'pais': 'MX'},   # Erika Elizalde
    1195190: {'sexo': 'hombre', 'pais': 'ES'},   # Manuel M. Velasco
    1805421: {'sexo': 'hombre', 'pais': 'ES'},   # Isaac Berrocal
    2876280: {'sexo': 'mujer',  'pais': 'ES'},   # Victoria Vázquez
    5467197: {'sexo': 'hombre', 'pais': None},   # Ignacio López
    2261043: {'sexo': 'hombre', 'pais': 'ES'},   # Diego Arjona
    2683614: {'sexo': 'mujer',  'pais': 'ES'},   # Amparo Aguilar
    2509031: {'sexo': 'hombre', 'pais': 'AR'},   # Pío Longo
    4163858: {'sexo': 'hombre', 'pais': 'PE'},   # Santiago Alvarado Ilarri
    1443072: {'sexo': 'hombre', 'pais': 'ES'},   # Ramón Salas
    3081311: {'sexo': 'hombre', 'pais': 'CO'},   # Juan A. Zapata
    5844744: {'sexo': 'hombre', 'pais': None},   # Alejandro Hernández
    3698547: {'sexo': 'mujer',  'pais': 'ES'},   # Carmina Balaguer
    572115:  {'sexo': 'hombre', 'pais': 'PR'},   # Ray Figueroa
    2427245: {'sexo': 'hombre', 'pais': 'MX'},   # Julio César Sánchez
    1634868: {'sexo': 'hombre', 'pais': 'ES'},   # José Manuel Colón
    572168:  {'sexo': 'hombre', 'pais': 'US'},   # Darren Kloomok
    1432104: {'sexo': 'hombre', 'pais': 'CO'},   # Alejandro Naranjo
    3763970: {'sexo': 'hombre', 'pais': 'ES'},   # Omar Al Abdul Razzak
    1027743: {'sexo': 'hombre', 'pais': 'ES'},   # Manuel J. García
    168769:  {'sexo': 'hombre', 'pais': 'DE'},   # Michael Maurer
    2194080: {'sexo': 'hombre', 'pais': 'US'},   # John Michael Boughn
    260267:  {'sexo': 'hombre', 'pais': 'AR'},   # Mateo Iribarren
    1184043: {'sexo': 'hombre', 'pais': 'MX'},   # Haroldo Fajardo
    1629878: {'sexo': 'mujer',  'pais': 'BR'},   # Carolina Kotscho
    1469682: {'sexo': 'hombre', 'pais': 'ES'},   # Daniel Desaloms
    1591887: {'sexo': 'hombre', 'pais': None},   # Armando López
    589119:  {'sexo': 'hombre', 'pais': 'CO'},   # Iván Ávila Dueñas
    2051782: {'sexo': 'hombre', 'pais': 'ES'},   # Xavi Sala
    2466544: {'sexo': 'hombre', 'pais': 'EC'},   # Ignacio Guarderas
    2532194: {'sexo': 'mujer',  'pais': 'ES'},   # Pilar Seijas
    2475701: {'sexo': 'hombre', 'pais': 'ES'},   # Luis Cintora
    3584087: {'sexo': 'hombre', 'pais': 'AR'},   # Camilo Zaffora
    3698687: {'sexo': 'hombre', 'pais': 'MX'},   # Carlos Aguillón
    2427035: {'sexo': 'mujer',  'pais': 'CO'},   # Yennifer Uribe Alzate
    3989209: {'sexo': 'hombre', 'pais': 'ES'},   # David Beltrán i Marí
    5054401: {'sexo': 'mujer',  'pais': 'ES'},   # Katherine Tort
    5605266: {'sexo': 'mujer',  'pais': 'PE'},   # Valeria Calmet
    2657529: {'sexo': 'mujer',  'pais': 'MX'},   # Leticia Castillo
    1649980: {'pais': 'ES'}                      # Ángela Gallardo Bernal
}


In [39]:
def aplicar_correcciones_personas_df(personas_df, correcciones_dict, verbose_aplicados=True):
    personas_df = personas_df.copy()
    aplicados, omitidos = [], []

    for person_id, valores in correcciones_dict.items():
        mask = personas_df['person_id'] == person_id
        if not mask.any():
            omitidos.append(person_id)
            continue

        if 'sexo' in valores:
            personas_df.loc[mask, 'sexo_final'] = valores['sexo']
            personas_df.loc[mask, 'fuente_sexo'] = 'manual'
        if 'pais' in valores:
            personas_df.loc[mask, 'pais_final'] = valores['pais']
            personas_df.loc[mask, 'fuente_pais'] = 'manual'

        aplicados.append(person_id)
        if verbose_aplicados:
            nombre = personas_df.loc[mask, 'nombre'].values[0]
            print(f"✓ {person_id} ({nombre}): {valores}")

    print(f"\n{len(aplicados)} correcciones aplicadas, {len(omitidos)} omitidas "
          f"(person_id no presente en este corpus)")
    return personas_df, aplicados, omitidos

if CORRECCIONES_MANUALES_PERSONAS:
    personas_df, aplicados, omitidos = aplicar_correcciones_personas_df(personas_df, CORRECCIONES_MANUALES_PERSONAS)


✓ 3517055 (José Corral Llorente): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2018522 (David Matamoros): {'sexo': 'hombre', 'pais': 'ES'}
✓ 3962124 (Marco Lagarde): {'sexo': 'hombre', 'pais': 'AR'}
✓ 1694893 (Pedro Martín Calero): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1980582 (Juma Fodde): {'sexo': 'hombre', 'pais': 'IT'}
✓ 3190577 (Jacob Santana): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2628918 (Sergio Dow): {'sexo': 'hombre', 'pais': 'CO'}
✓ 2018520 (Ángeles Hernández): {'sexo': 'mujer', 'pais': 'ES'}
✓ 2549328 (Irene Iborra Rizo): {'sexo': 'mujer', 'pais': 'ES'}
✓ 1115313 (Javier Espada): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1525399 (Ida Cuéllar): {'sexo': 'hombre', 'pais': 'BO'}
✓ 3989209 (David Beltrán i Marí): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1533451 (Laura Baumeister): {'sexo': 'mujer', 'pais': 'NI'}
✓ 1091827 (Diego Yaker): {'sexo': 'hombre', 'pais': 'AR'}
✓ 2568321 (Johanne Gómez Terrero): {'sexo': 'mujer', 'pais': 'DO'}
✓ 4206690 (Tabatta Salinas Caballero): {'sexo': 'mujer', 'pais': 'MX'}
✓ 

In [42]:
pendientes = personas_df[personas_df['sexo_final'].isna() | personas_df['pais_final'].isna()].copy()
pendientes = pendientes.sort_values('nombre')

print(f"# {len(pendientes)} personas con algún dato faltante -- rellena 'sexo'/'pais' donde lo sepas, "
      "deja None donde no")
print("CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {")
for _, row in pendientes.iterrows():
    pid = int(row['person_id']) if pd.notna(row['person_id']) else None
    imdb_id = imdb_id_cache.get(str(pid)) if pid is not None else None
    link_imdb = f" | imdb.com/es/name/{imdb_id}/" if imdb_id else ""
    print(f"    {pid}: {{'sexo': None, 'pais': None}},  "
          f"# {row['nombre']} (sexo={row['sexo_final']}, pais={row['pais_final']}){link_imdb}")
print("}")

# 1312 personas con algún dato faltante -- rellena 'sexo'/'pais' donde lo sepas, deja None donde no
CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {
    4271430: {'sexo': None, 'pais': None},  # Abel Alfonso (sexo=nan, pais=nan)
    1049263: {'sexo': None, 'pais': None},  # Abel García Roure (sexo=nan, pais=nan) | imdb.com/es/name/nm1071819/
    2806169: {'sexo': None, 'pais': None},  # Abigail Schaaff (sexo=nan, pais=nan)
    1413007: {'sexo': None, 'pais': None},  # Adolfo Dufour (sexo=hombre, pais=nan) | imdb.com/es/name/nm1580691/
    4180516: {'sexo': None, 'pais': None},  # Adolfo Moreno (sexo=nan, pais=nan)
    4181357: {'sexo': None, 'pais': None},  # Adrian Varela (sexo=nan, pais=nan)
    3959299: {'sexo': None, 'pais': None},  # Adrià Roca (sexo=hombre, pais=nan)
    5048846: {'sexo': None, 'pais': None},  # Adrián García (sexo=nan, pais=nan)
    5686508: {'sexo': None, 'pais': None},  # Adrián Guanche (sexo=nan, pais=nan)
    1802332: {'sexo': None, 'pais': None},  # Adrián Orr (se

In [43]:
PAISES_LATAM_NOMBRES = {
    'México', 'Argentina', 'Brasil', 'Chile', 'Colombia', 'Perú', 'Venezuela',
    'Uruguay', 'Paraguay', 'Bolivia', 'Ecuador', 'Costa Rica', 'Panamá',
    'Guatemala', 'Honduras', 'El Salvador', 'Nicaragua', 'República Dominicana',
    'Cuba', 'Puerto Rico',
}

def extraer_paises_icaa(nacionalidad_str):
    if pd.isna(nacionalidad_str) or not nacionalidad_str:
        return []
    return [re.sub(r'\s*\([\d.]+%\)\s*$', '', p).strip() for p in nacionalidad_str.split(';')]

def tiene_pais_latam_icaa(nacionalidad_str):
    return bool(set(extraer_paises_icaa(nacionalidad_str)) & PAISES_LATAM_NOMBRES)


icaa_peliculas['es_latam'] = icaa_peliculas['nacionalidad_paises_icaa'].apply(tiene_pais_latam_icaa)
peliculas_latam_ids = set(icaa_peliculas.loc[icaa_peliculas['es_latam'], 'icaa_id'].dropna())

person_ids_latam = set(
    credits_df.loc[credits_df['icaa_id'].isin(peliculas_latam_ids), 'person_id'].dropna()
)

print(f"Películas LATAM: {len(peliculas_latam_ids)} / {icaa_peliculas['icaa_id'].notna().sum()}")
print(f"Personas relevantes a LATAM: {len(person_ids_latam)}")

Películas LATAM: 144 / 1775
Personas relevantes a LATAM: 247


In [44]:
pendientes_latam = personas_df[
    (personas_df['sexo_final'].isna() | personas_df['pais_final'].isna())
    & personas_df['person_id'].isin(person_ids_latam)
].copy().sort_values('nombre')

print(f"# {len(pendientes_latam)} personas LATAM con algún dato faltante "
      f"(de {(personas_df['sexo_final'].isna() | personas_df['pais_final'].isna()).sum()} totales) "
      "-- rellena 'sexo'/'pais' donde lo sepas, deja None donde no")
print("CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {")
for _, row in pendientes_latam.iterrows():
    pid = int(row['person_id']) if pd.notna(row['person_id']) else None
    imdb_id = imdb_id_cache.get(str(pid)) if pid is not None else None
    link_imdb = f" | imdb.com/es/name/{imdb_id}/" if imdb_id else ""
    print(f"    {pid}: {{'sexo': None, 'pais': None}},  "
          f"# {row['nombre']} (sexo={row['sexo_final']}, pais={row['pais_final']}){link_imdb}")
print("}")

# 29 personas LATAM con algún dato faltante (de 1312 totales) -- rellena 'sexo'/'pais' donde lo sepas, deja None donde no
CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {
    5844744: {'sexo': None, 'pais': None},  # Alejandro Hernández (sexo=hombre, pais=nan)
    3108914: {'sexo': None, 'pais': None},  # Ana de Alva (sexo=nan, pais=nan)
    3840554: {'sexo': None, 'pais': None},  # Araceli Gonda (sexo=nan, pais=nan)
    3195500: {'sexo': None, 'pais': None},  # Beatriz Iznaola (sexo=nan, pais=nan)
    5727167: {'sexo': None, 'pais': None},  # Carmen López (sexo=mujer, pais=nan)
    3212692: {'sexo': None, 'pais': None},  # Cristina Broquetas (sexo=nan, pais=nan)
    2852263: {'sexo': None, 'pais': None},  # Daniel Monedero (sexo=nan, pais=nan)
    5736743: {'sexo': None, 'pais': None},  # David Pascual (sexo=hombre, pais=nan)
    5597231: {'sexo': None, 'pais': None},  # Diego Martínez (sexo=hombre, pais=nan)
    2865693: {'sexo': None, 'pais': None},  # Francisco Arnal (sexo=nan, pais=nan)


In [ ]:
# 29 personas LATAM con algún dato faltante (de 1312 totales) -- rellena 'sexo'/'pais' donde lo sepas, deja None donde no
CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {
    5844744: {'sexo': 'hombre', 'pais': 'CU'},  # Alejandro Hernández (sexo=hombre, pais=nan)
    3108914: {'sexo': 'mujer', 'pais': 'ES'},  # Ana de Alva (sexo=nan, pais=nan)
    3840554: {'sexo': 'mujer', 'pais': 'ES'},  # Araceli Gonda (sexo=nan, pais=nan)
    3195500: {'sexo': 'mujer', 'pais': 'ES'},  # Beatriz Iznaola (sexo=nan, pais=nan)
    5727167: {'sexo': 'mujer', 'pais': 'ES'},  # Carmen López (sexo=mujer, pais=nan)
    3212692: {'sexo': 'mujer', 'pais': 'ES'},  # Cristina Broquetas (sexo=nan, pais=nan)
    2852263: {'sexo': 'hombre', 'pais': 'ES'},  # Daniel Monedero (sexo=nan, pais=nan)
    5736743: {'sexo': 'hombre', 'pais': 'ES'},  # David Pascual (sexo=hombre, pais=nan)
    5597231: {'sexo': 'hombre', 'pais': 'ES'},  # Diego Martínez (sexo=hombre, pais=nan)
    2865693: {'sexo': 'hombre', 'pais': 'ES'},  # Francisco Arnal (sexo=nan, pais=nan)
    1835761: {'sexo': 'hombre', 'pais': 'UY'},  # Gustavo Hernández (sexo=hombre, pais=nan)
    3276081: {'sexo': 'mujer', 'pais': 'ES'},  # Haizea Pastor (sexo=nan, pais=nan)
    4954090: {'sexo': 'hombre', 'pais': 'ES'},  # Iker Álvarez (sexo=nan, pais=nan)
    5295734: {'sexo': 'mujer', 'pais': 'ES'},  # Ingride Santos (sexo=nan, pais=nan)
    2437353: {'sexo': 'mujer', 'pais': 'ES'},  # Inés Silva (sexo=nan, pais=nan)
    4076684: {'sexo': 'hombre', 'pais': 'ES'},  # José Ortuño (sexo=nan, pais=nan)
    4916124: {'sexo': 'hombre', 'pais': 'ES'},  # José Pérez (sexo=nan, pais=nan)
    1564500: {'sexo': None, 'pais': None},  # Juan Clemente Prosper (sexo=nan, pais=nan)
    1097734: {'sexo': 'hombre', 'pais': 'ES'},  # Juan Luis de No (sexo=nan, pais=nan) | imdb.com/es/name/nm1907971/
    3862769: {'sexo': None, 'pais': None},  # Liliana Torres (sexo=mujer, pais=nan)
    3374272: {'sexo': 'mujer', 'pais': 'ES'},  # Lucía Herrera (sexo=mujer, pais=nan)
    2915542: {'sexo': 'mujer', 'pais': 'ES'},  # Lula Gómez (sexo=nan, pais=nan)
    5065139: {'sexo': 'hombre', 'pais': 'AR'},  # Matías Goldberg (sexo=nan, pais=nan)
    3016177: {'sexo': 'mujer', 'pais': 'ES'},  # Mercedes Rodrigo (sexo=mujer, pais=nan)
    2494026: {'sexo': 'mujer', 'pais': 'ES'},  # Pepa G. Ramos (sexo=nan, pais=nan)
    2915538: {'sexo': 'mujer', 'pais': 'ES'},  # Susana Guardiola (sexo=nan, pais=nan)
    2075885: {'sexo': 'mujer', 'pais': 'ES'},  # Tirma Pérez (sexo=nan, pais=nan)
    4534713: {'sexo': 'mujer', 'pais': 'ES'},  # Yurre Ugarte (sexo=nan, pais=nan)
    4983818: {'sexo': 'hombre', 'pais': 'ES'},  # Ángel Gómez (sexo=hombre, pais=nan)
}

Juan Prosper no es Juan Clemente Prosper (1564500) - Eliminar

In [47]:
person_search_cache['Juan Prosper'] = {"person_id": None, "nombre_tmdb": None}
guardar_cache_json(PERSON_SEARCH_CACHE, person_search_cache)

In [49]:
pendientes_latam = personas_df[
    (personas_df['sexo_final'].isna() | personas_df['pais_final'].isna())
    & personas_df['person_id'].isin(person_ids_latam)
].copy().sort_values('nombre')

print(f"# {len(pendientes_latam)} personas LATAM con algún dato faltante "
      f"(de {(personas_df['sexo_final'].isna() | personas_df['pais_final'].isna()).sum()} totales) "
      "-- rellena 'sexo'/'pais' donde lo sepas, deja None donde no")
print("CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {")
for _, row in pendientes_latam.iterrows():
    pid = int(row['person_id']) if pd.notna(row['person_id']) else None
    imdb_id = imdb_id_cache.get(str(pid)) if pid is not None else None
    link_imdb = f" | imdb.com/es/name/{imdb_id}/" if imdb_id else ""
    print(f"    {pid}: {{'sexo': None, 'pais': None}},  "
          f"# {row['nombre']} (sexo={row['sexo_final']}, pais={row['pais_final']}){link_imdb}")
print("}")

# 2 personas LATAM con algún dato faltante (de 1285 totales) -- rellena 'sexo'/'pais' donde lo sepas, deja None donde no
CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {
    1564500: {'sexo': None, 'pais': None},  # Juan Clemente Prosper (sexo=nan, pais=nan)
    3862769: {'sexo': None, 'pais': None},  # Liliana Torres (sexo=mujer, pais=nan)
}


Update

In [46]:
CORRECCIONES_MANUALES_PERSONAS.update({
    pid: {k: v for k, v in valores.items() if v is not None}
    for pid, valores in CORRECCIONES_MANUALES_PERSONAS_NUEVAS.items()
    if any(v is not None for v in valores.values())
})

personas_df, aplicados, omitidos = aplicar_correcciones_personas_df(personas_df, CORRECCIONES_MANUALES_PERSONAS)

✓ 3517055 (José Corral Llorente): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2018522 (David Matamoros): {'sexo': 'hombre', 'pais': 'ES'}
✓ 3962124 (Marco Lagarde): {'sexo': 'hombre', 'pais': 'AR'}
✓ 1694893 (Pedro Martín Calero): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1980582 (Juma Fodde): {'sexo': 'hombre', 'pais': 'IT'}
✓ 3190577 (Jacob Santana): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2628918 (Sergio Dow): {'sexo': 'hombre', 'pais': 'CO'}
✓ 2018520 (Ángeles Hernández): {'sexo': 'mujer', 'pais': 'ES'}
✓ 2549328 (Irene Iborra Rizo): {'sexo': 'mujer', 'pais': 'ES'}
✓ 1115313 (Javier Espada): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1525399 (Ida Cuéllar): {'sexo': 'hombre', 'pais': 'BO'}
✓ 3989209 (David Beltrán i Marí): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1533451 (Laura Baumeister): {'sexo': 'mujer', 'pais': 'NI'}
✓ 1091827 (Diego Yaker): {'sexo': 'hombre', 'pais': 'AR'}
✓ 2568321 (Johanne Gómez Terrero): {'sexo': 'mujer', 'pais': 'DO'}
✓ 4206690 (Tabatta Salinas Caballero): {'sexo': 'mujer', 'pais': 'MX'}
✓ 

In [50]:
# 2 personas LATAM con algún dato faltante (de 1285 totales) -- rellena 'sexo'/'pais' donde lo sepas, deja None donde no
CORRECCIONES_MANUALES_PERSONAS_NUEVAS = {
    3862769: {'sexo': 'mujer', 'pais': 'ES'},  # Liliana Torres (sexo=mujer, pais=nan)
}

In [51]:
CORRECCIONES_MANUALES_PERSONAS.update({
    pid: {k: v for k, v in valores.items() if v is not None}
    for pid, valores in CORRECCIONES_MANUALES_PERSONAS_NUEVAS.items()
    if any(v is not None for v in valores.values())
})

personas_df, aplicados, omitidos = aplicar_correcciones_personas_df(personas_df, CORRECCIONES_MANUALES_PERSONAS)

✓ 3517055 (José Corral Llorente): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2018522 (David Matamoros): {'sexo': 'hombre', 'pais': 'ES'}
✓ 3962124 (Marco Lagarde): {'sexo': 'hombre', 'pais': 'AR'}
✓ 1694893 (Pedro Martín Calero): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1980582 (Juma Fodde): {'sexo': 'hombre', 'pais': 'IT'}
✓ 3190577 (Jacob Santana): {'sexo': 'hombre', 'pais': 'ES'}
✓ 2628918 (Sergio Dow): {'sexo': 'hombre', 'pais': 'CO'}
✓ 2018520 (Ángeles Hernández): {'sexo': 'mujer', 'pais': 'ES'}
✓ 2549328 (Irene Iborra Rizo): {'sexo': 'mujer', 'pais': 'ES'}
✓ 1115313 (Javier Espada): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1525399 (Ida Cuéllar): {'sexo': 'hombre', 'pais': 'BO'}
✓ 3989209 (David Beltrán i Marí): {'sexo': 'hombre', 'pais': 'ES'}
✓ 1533451 (Laura Baumeister): {'sexo': 'mujer', 'pais': 'NI'}
✓ 1091827 (Diego Yaker): {'sexo': 'hombre', 'pais': 'AR'}
✓ 2568321 (Johanne Gómez Terrero): {'sexo': 'mujer', 'pais': 'DO'}
✓ 4206690 (Tabatta Salinas Caballero): {'sexo': 'mujer', 'pais': 'MX'}
✓ 

In [52]:
print(f"Cobertura final tras correcciones manuales:")
print(f"Sexo: {personas_df['sexo_final'].notna().sum()} / {len(personas_df)}")
print(f"País: {personas_df['pais_final'].notna().sum()} / {len(personas_df)}")
print(f"\nFuente del dato de sexo:\n{personas_df['fuente_sexo'].value_counts(dropna=False)}")
print(f"\nFuente del dato de país:\n{personas_df['fuente_pais'].value_counts(dropna=False)}")


Cobertura final tras correcciones manuales:
Sexo: 1173 / 2262
País: 988 / 2262

Fuente del dato de sexo:
fuente_sexo
NaN         1089
tmdb         829
wikidata     177
manual       167
Name: count, dtype: int64

Fuente del dato de país:
fuente_pais
None               1271
tmdb_heuristico     561
wikidata            262
manual              168
Name: count, dtype: int64


In [53]:
def personas_df_pendientes(personas_df):
    pendientes = personas_df[personas_df['sexo_final'].isna() | personas_df['pais_final'].isna()]
    print(f"{len(pendientes)} personas con algún dato faltante de {len(personas_df)} totales")
    for _, row in pendientes.iterrows():
        print(f"  person_id={row['person_id']}: {row['nombre']} "
              f"(sexo={row['sexo_final']}, pais={row['pais_final']})")
    return pendientes

pendientes_personas = personas_df_pendientes(personas_df)


1284 personas con algún dato faltante de 2262 totales
  person_id=1930872.0: Ferrán Navarro-Beltrán (sexo=hombre, pais=nan)
  person_id=1114075.0: Vicente Pérez Herrero (sexo=nan, pais=nan)
  person_id=4748222.0: Jose Ramón Fernández Pinazo (sexo=nan, pais=nan)
  person_id=2434913.0: César Souto Vilanova (sexo=nan, pais=nan)
  person_id=2434919.0: Cristina Liz Graña (sexo=nan, pais=nan)
  person_id=5349292.0: Ramón Pineda (sexo=nan, pais=nan)
  person_id=2522953.0: Patricia Franquesa (sexo=nan, pais=nan)
  person_id=1284828.0: Juan Barrero (sexo=nan, pais=nan)
  person_id=5955689.0: Moisés Garrido (sexo=nan, pais=nan)
  person_id=3567270.0: Evgeny Yablokov (sexo=nan, pais=nan)
  person_id=3778185.0: Nick Igea (sexo=nan, pais=nan)
  person_id=1624337.0: Pau Faus (sexo=nan, pais=nan)
  person_id=3884067.0: Júlia R. Aymar (sexo=nan, pais=nan)
  person_id=4382575.0: Ana L. Martín (sexo=nan, pais=nan)
  person_id=5694090.0: Marta García (sexo=nan, pais=nan)
  person_id=1105225.0: Héctor Cla

## 12. Construcción final -- solo dimensiones (`direccion`, `guion`)

Aquí SÍ se puede finalizar: `direccion`/`guion` son tablas de personas con
`sexo`/`país`, no dependen de `pelicula_id`. Lo que NO se construye todavía
son `direccion_peliculas`/`guion_peliculas` -- esas referencian películas,
y `pelicula_id` (la clave surrogada real) todavía no existe; se asigna en
el notebook de tablas finales. Construirlas aquí con `icaa_id`/`tmdb_id`
como referencia temporal sería una capa de indirección frágil que habría
que deshacer después.

In [54]:
def construir_dim_personas(rol, prefijo):
    creditos_rol = credits_df[credits_df['rol'] == rol].copy()

    con_id = creditos_rol[creditos_rol['person_id'].notna()].drop_duplicates(subset=['person_id'])
    sin_id = creditos_rol[creditos_rol['person_id'].isna()].drop_duplicates(subset=['nombre'])

    personas_unicas = (
        con_id[['person_id']]
        .merge(
            personas_df[['person_id', 'nombre', 'sexo_final', 'fuente_sexo',
                         'pais_final', 'fuente_pais']],
            on='person_id', how='left', validate='many_to_one'
        )
    )
    personas_unicas['nombre_credito_original'] = None

    personas_sin_match = (
        sin_id[['nombre']].rename(columns={'nombre': 'nombre_credito_original'})
    )
    personas_sin_match['person_id'] = None
    personas_sin_match['nombre'] = personas_sin_match['nombre_credito_original']
    personas_sin_match['sexo_final'] = None
    personas_sin_match['fuente_sexo'] = None
    personas_sin_match['pais_final'] = None
    personas_sin_match['fuente_pais'] = None

    dim = pd.concat([personas_unicas, personas_sin_match], ignore_index=True).reset_index(drop=True)
    ids_generados = [f"{prefijo}_{i:04d}" for i in range(1, len(dim) + 1)]
    dim.insert(0, f"{prefijo}_id", ids_generados)
    dim = dim.rename(columns={'sexo_final': 'sexo', 'pais_final': 'pais'})
    return dim


direccion = construir_dim_personas('director', 'dir')
guion = construir_dim_personas('guionista', 'gui')

print(f"direccion: {len(direccion)} personas únicas")
print(f"guion: {len(guion)} personas únicas")


direccion: 1649 personas únicas
guion: 2323 personas únicas


### Correcciones manuales -- personas sin `person_id`

Igual que antes, pero ahora la clave es `dir_id`/`gui_id` en vez de
`person_id`, porque estas personas no tienen `person_id` en absoluto
(TMDB nunca las encontró).

In [60]:
def personas_pendientes(dim_df, prefijo):
    pendientes = dim_df[dim_df['sexo'].isna() | dim_df['pais'].isna()]
    print(f"{prefijo}: {len(pendientes)} personas con algún dato faltante de {len(dim_df)} totales")
    for _, row in pendientes.iterrows():
        nombre = row['nombre'] if pd.notna(row['nombre']) else row.get('nombre_credito_original')
        print(f"  {row[f'{prefijo}_id']}: {nombre} (sexo={row['sexo']}, pais={row['pais']}, person_id={row['person_id']})")
    return pendientes

print("=== DIRECCIÓN ===")
pendientes_direccion = personas_pendientes(direccion, 'dir')
print("\n=== GUION ===")
pendientes_guion = personas_pendientes(guion, 'gui')


=== DIRECCIÓN ===
dir: 954 personas con algún dato faltante de 1649 totales
  dir_0002: Ferrán Navarro-Beltrán (sexo=hombre, pais=nan, person_id=1930872.0)
  dir_0003: Vicente Pérez Herrero (sexo=nan, pais=nan, person_id=1114075.0)
  dir_0006: César Souto Vilanova (sexo=nan, pais=nan, person_id=2434913.0)
  dir_0008: Patricia Franquesa (sexo=nan, pais=nan, person_id=2522953.0)
  dir_0012: Evgeny Yablokov (sexo=nan, pais=nan, person_id=3567270.0)
  dir_0013: Nick Igea (sexo=nan, pais=nan, person_id=3778185.0)
  dir_0016: Pau Faus (sexo=nan, pais=nan, person_id=1624337.0)
  dir_0017: Ana L. Martín (sexo=nan, pais=nan, person_id=4382575.0)
  dir_0018: Héctor Claramunt (sexo=hombre, pais=nan, person_id=1105225.0)
  dir_0019: Valeriano López (sexo=nan, pais=nan, person_id=2445361.0)
  dir_0021: Alberto Martos (sexo=nan, pais=nan, person_id=5702959.0)
  dir_0023: Elena Manrique (sexo=mujer, pais=nan, person_id=17000.0)
  dir_0024: Chiara Avesani (sexo=nan, pais=nan, person_id=3217186.0)
  di

In [62]:
correccion = {
    'nombre': 'Juan Prosper',
    'person_id': None,
    'sexo': 'hombre',
    'fuente_sexo': 'manual',
    'pais': 'MX',
    'fuente_pais': 'manual',
}

mask = guion['gui_id'] == 'gui_1440'
for columna, valor in correccion.items():
    guion.loc[mask, columna] = valor

In [63]:
"""
Formato: {id: {'sexo': 'mujer'/'hombre'/'no_binario', 'pais': 'XX' (ISO alpha-2)}
"""

CORRECCIONES_MANUALES_DIR_GUI = {
    # 'dir_0007': {'sexo': 'mujer', 'pais': 'AR'},
    # 'gui_0012': {'pais': 'MX'},
}


def aplicar_correcciones_dim(dim_df, prefijo, correcciones_dict):
    dim_df = dim_df.copy()
    id_col = f"{prefijo}_id"
    aplicados = []
    for id_persona, valores in correcciones_dict.items():
        if id_persona not in dim_df[id_col].values:
            print(f"⚠ {id_persona} no encontrado en la tabla, se omite")
            continue
        mask = dim_df[id_col] == id_persona
        if 'sexo' in valores:
            dim_df.loc[mask, 'sexo'] = valores['sexo']
            dim_df.loc[mask, 'fuente_sexo'] = 'manual'
        if 'pais' in valores:
            dim_df.loc[mask, 'pais'] = valores['pais']
            dim_df.loc[mask, 'fuente_pais'] = 'manual'
        aplicados.append(id_persona)
    print(f"{len(aplicados)} correcciones aplicadas de {len(correcciones_dict)}")
    return dim_df


if CORRECCIONES_MANUALES_DIR_GUI:
    direccion = aplicar_correcciones_dim(direccion, 'dir', CORRECCIONES_MANUALES_DIR_GUI)
    guion = aplicar_correcciones_dim(guion, 'gui', CORRECCIONES_MANUALES_DIR_GUI)


## 13. Créditos intermedios (para el notebook de tablas finales)

Tabla puente **provisional**: `icaa_id`/`tmdb_id` (referencia a película,
resoluble vía `icaa_peliculas.csv` ya actualizado) + `dir_id`/`gui_id`
(referencia a persona). El notebook de tablas finales la lee, asigna
`pelicula_id` real (uniendo por `icaa_id`/`tmdb_id`) y con eso construye
`direccion_peliculas`/`guion_peliculas` definitivas -- esta tabla no se
sube a MySQL, es solo un CSV de paso.

In [64]:
def construir_creditos_intermedios(rol, prefijo, dim):
    creditos_rol = credits_df[credits_df['rol'] == rol].copy()

    con_id = creditos_rol[creditos_rol['person_id'].notna()].drop_duplicates(
        subset=['icaa_id', 'tmdb_id', 'person_id']
    )
    sin_id = creditos_rol[creditos_rol['person_id'].isna()].drop_duplicates(
        subset=['icaa_id', 'tmdb_id', 'nombre']
    )

    con_id = con_id.merge(
        dim[dim['person_id'].notna()][[f"{prefijo}_id", 'person_id']],
        on='person_id', how='left', validate='many_to_one'
    )
    sin_id = sin_id.merge(
        dim[dim['person_id'].isna()][[f"{prefijo}_id", 'nombre_credito_original']],
        left_on='nombre', right_on='nombre_credito_original', how='left'
    )

    creditos = pd.concat([con_id, sin_id], ignore_index=True)[
        ['icaa_id', 'tmdb_id', f"{prefijo}_id", 'fuente_credito', 'metodo_match_persona']
    ].reset_index(drop=True)
    return creditos


creditos_direccion_intermedio = construir_creditos_intermedios('director', 'dir', direccion)
creditos_guion_intermedio = construir_creditos_intermedios('guionista', 'gui', guion)

print(f"creditos_direccion_intermedio: {len(creditos_direccion_intermedio)} filas")
print(f"creditos_guion_intermedio: {len(creditos_guion_intermedio)} filas")

revisar_matches = pd.concat([creditos_direccion_intermedio, creditos_guion_intermedio])
revisar_matches = revisar_matches[revisar_matches['metodo_match_persona'] == 'busqueda_nombre_aproximada']
print(f"\n⚠ Créditos a revisar por match aproximado (nombre no coincide exacto): {len(revisar_matches)}")


creditos_direccion_intermedio: 2159 filas
creditos_guion_intermedio: 2992 filas

⚠ Créditos a revisar por match aproximado (nombre no coincide exacto): 310


## 14. Exportar

In [65]:
# direccion/guion sí van a MySQL -- son tablas finales, no dependen de pelicula_id
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)
direccion.to_sql('direccion', engine, if_exists='replace', index=False)
guion.to_sql('guion', engine, if_exists='replace', index=False)
print("✓ Exportado a MySQL: direccion, guion")
engine.dispose()
print("MySQL conexión cerrada.")


✓ Exportado a MySQL: direccion, guion
MySQL conexión cerrada.


In [66]:
direccion.to_csv(CSV_DIR / "direccion.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
guion.to_csv(CSV_DIR / "guion.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)

# Créditos intermedios -- SOLO CSV, no van a MySQL todavía (esquema no final)
creditos_direccion_intermedio.to_csv(CSV_DIR / "creditos_direccion_intermedio.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
creditos_guion_intermedio.to_csv(CSV_DIR / "creditos_guion_intermedio.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)

print("✓ Exportado a CSV:")
for nombre in ['direccion', 'guion', 'creditos_direccion_intermedio', 'creditos_guion_intermedio']:
    ruta = CSV_DIR / f"{nombre}.csv"
    print(f"  {ruta} ({ruta.stat().st_size / 1024:.1f} KB)")


✓ Exportado a CSV:
  ..\3 - csv\direccion.csv (111.0 KB)
  ..\3 - csv\guion.csv (154.0 KB)
  ..\3 - csv\creditos_direccion_intermedio.csv (112.0 KB)
  ..\3 - csv\creditos_guion_intermedio.csv (153.3 KB)
